In [2]:
# Cell 1 (revised): Load artifacts from custom base_dir (Windows path aware) & lock feature list
import os
from pathlib import Path
import json
import pandas as pd
import numpy as np
import textwrap
import glob

# ---- EDIT ONLY IF YOUR FILES ARE IN A DIFFERENT FOLDER ----
# You told me they're in:
base_dir = r"D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected"

print("Base dir (user-provided):", base_dir)
base_path = Path(base_dir)
if not base_path.exists():
    print("\nWARNING: Provided base_dir does not exist on this environment (that's expected if you're running locally).")
    print("I'll also attempt to search current working dir and /mnt/data for likely files.")
else:
    print("Base dir exists. Scanning for stage2 CSVs and feature config...")

# helper to build candidate paths using base_dir
def candidates_with_base(base):
    b = Path(base)
    return {
        "X_train": [str(b / fname) for fname in ("X_train_stage2.csv", "train_stage2_final.csv", "X_train.csv", "X_train_stage2_v2.csv")],
        "y_train": [str(b / fname) for fname in ("y_train_stage2.csv", "y_train.csv", "y_train_stage2_v2.csv")],
        "X_val":   [str(b / fname) for fname in ("X_val_stage2.csv", "val_stage2_final.csv", "X_val.csv")],
        "y_val":   [str(b / fname) for fname in ("y_val_stage2.csv", "y_val.csv")],
        "X_test":  [str(b / fname) for fname in ("X_test_stage2.csv", "test_stage2_final.csv", "X_test.csv")],
        "y_test":  [str(b / fname) for fname in ("y_test_stage2.csv", "y_test.csv")],
        "feat_cfg":[str(b / fname) for fname in ("stage2_feature_config.json", "stage2_feature_config*.json")]
    }

# primary candidates: explicit base_dir
primary_candidates = candidates_with_base(base_dir)

# fallback candidates (previous)
fallback_candidates = {
    "X_train": ["X_train_stage2.csv", "train_stage2_final.csv", "/mnt/data/train_stage2_final.csv", "/mnt/data/X_train_stage2.csv"],
    "y_train": ["y_train_stage2.csv", "/mnt/data/y_train_stage2.csv"],
    "X_val":   ["X_val_stage2.csv", "val_stage2_final.csv", "/mnt/data/val_stage2_final.csv", "/mnt/data/X_val_stage2.csv"],
    "y_val":   ["y_val_stage2.csv", "/mnt/data/y_val_stage2.csv"],
    "X_test":  ["X_test_stage2.csv", "test_stage2_final.csv", "/mnt/data/test_stage2_final.csv", "/mnt/data/X_test_stage2.csv"],
    "y_test":  ["y_test_stage2.csv", "/mnt/data/y_test_stage2.csv"],
    "feat_cfg": ["stage2_feature_config.json", "stage2_feature_config*.json", "/mnt/data/stage2_feature_config.json"]
}

# merge: primary first, then fallback, then search heuristics
candidates = {}
for key in primary_candidates.keys():
    c = []
    c.extend(primary_candidates.get(key, []))
    c.extend(fallback_candidates.get(key, []))
    # add any CSVs in base_dir that contain the key token
    if base_path.exists():
        token = key.split('_')[-1]  # e.g., train, val, test, feat_cfg
        for p in base_path.glob(f"*{token}*"):
            c.append(str(p))
        # also add all csv/json in base_dir as last resort
        for p in base_path.glob("*.csv"):
            c.append(str(p))
        for p in base_path.glob("*.json"):
            c.append(str(p))
    # Add /mnt/data wildcards
    c.extend(glob.glob("/mnt/data/*.csv"))
    c.extend(glob.glob("/mnt/data/*.json"))
    # dedupe while preserving order
    seen = set()
    dedup = []
    for x in c:
        if x not in seen:
            dedup.append(x)
            seen.add(x)
    candidates[key] = dedup

# find_first_existing allowing globs in entries
def find_first_existing(path_list):
    for p in path_list:
        try:
            # expand user and vars
            p_exp = os.path.expanduser(os.path.expandvars(p))
            # if contains wildcard
            if "*" in p_exp:
                matches = glob.glob(p_exp)
                if matches:
                    return matches[0]
                continue
            if os.path.exists(p_exp):
                return p_exp
        except Exception:
            continue
    return None

found = {}
for k, paths in candidates.items():
    found_path = find_first_existing(paths)
    found[k] = found_path

print("\nFile discovery tentative report (search order: base_dir -> fallback -> /mnt/data):")
for k, p in found.items():
    print(f"  {k}: {p if p is not None else 'NOT FOUND'}")

missing_required = [k for k in ("X_train","y_train","X_val","y_val","X_test","y_test") if not found.get(k)]
if missing_required:
    raise FileNotFoundError(
        "Missing required dataset files in the discovered locations. "
        "Please either: (A) run this notebook where the 'base_dir' path is accessible, or (B) update `base_dir` above to the folder path containing your CSVs, or (C) copy the CSVs into the notebook environment's /mnt/data. "
        f"Missing keys: {missing_required}"
    )

# Load CSVs
X_train = pd.read_csv(found["X_train"])
y_train = pd.read_csv(found["y_train"]).squeeze()
X_val   = pd.read_csv(found["X_val"])
y_val   = pd.read_csv(found["y_val"]).squeeze()
X_test  = pd.read_csv(found["X_test"])
y_test  = pd.read_csv(found["y_test"]).squeeze()

print("\nLoaded shapes:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"  X_test:  {X_test.shape}, y_test:  {y_test.shape}")

# Load feature config if found
feat_cfg = None
if found.get("feat_cfg"):
    try:
        with open(found["feat_cfg"]) as f:
            feat_cfg = json.load(f)
        print(f"\nLoaded feature config from: {found['feat_cfg']}")
    except Exception as e:
        print(f"\nCould not load feature config at {found['feat_cfg']}: {e}")

# Basic sanity report (no display() to keep env-agnostic)
def quick_report(df, name, nrows=3):
    print(f"\n{name} head (first {nrows} rows):")
    print(df.head(nrows).to_string(index=False))
    print(f"\n{name} dtypes counts:")
    print(df.dtypes.value_counts().to_string())
    obj_cols = df.select_dtypes(include=['object']).columns.tolist()
    if obj_cols:
        print(f"  -> {len(obj_cols)} object dtype columns in {name}: {obj_cols[:20]}")
    else:
        print(f"  -> no object dtype columns in {name}")

quick_report(X_train, "X_train")
quick_report(X_val, "X_val")
quick_report(X_test, "X_test")

# Feature list checks
features_train = list(X_train.columns)
features_val   = list(X_val.columns)
features_test  = list(X_test.columns)

print("\nFeature counts:")
print(f"  train: {len(features_train)}, val: {len(features_val)}, test: {len(features_test)}")

set_train = set(features_train)
set_val = set(features_val)
set_test = set(features_test)

common = set_train & set_val & set_test
only_in_train = set_train - (set_val | set_test)
only_in_val = set_val - (set_train | set_test)
only_in_test = set_test - (set_train | set_val)

print(f"\nCommon features across all splits: {len(common)}")
if len(common) <= 60:
    print(sorted(common))

if only_in_train or only_in_val or only_in_test:
    print("\nFeature mismatches (detailed):")
    if only_in_train:
        print(f"  Only in TRAIN ({len(only_in_train)}): {sorted(list(only_in_train))[:30]}")
    if only_in_val:
        print(f"  Only in VAL   ({len(only_in_val)}): {sorted(list(only_in_val))[:30]}")
    if only_in_test:
        print(f"  Only in TEST  ({len(only_in_test)}): {sorted(list(only_in_test))[:30]}")

# Assert same 32 features as required
expected_n_features = 32
assert len(common) == expected_n_features, (
    f"ASSERTION FAILED: Expected {expected_n_features} identical features across splits, "
    f"but found {len(common)} common features. See mismatch printout above."
)

# Ensure no object dtypes in Xs
obj_train = X_train.select_dtypes(include=['object']).columns.tolist()
obj_val = X_val.select_dtypes(include=['object']).columns.tolist()
obj_test = X_test.select_dtypes(include=['object']).columns.tolist()
if obj_train or obj_val or obj_test:
    raise AssertionError(
        f"ASSERTION FAILED: Object dtypes found. train: {obj_train}, val: {obj_val}, test: {obj_test}. "
        "Convert/encode these before modeling or ensure the CSVs export numeric-only features."
    )

locked_features = sorted(list(common))
print("\nFEATURE LIST LOCKED (sorted):")
print(locked_features)

if feat_cfg:
    cfg_features = feat_cfg.get("features") or feat_cfg.get("feature_list") or feat_cfg.get("columns")
    if cfg_features:
        cfg_set = set(cfg_features)
        if cfg_set != common:
            print("\nWARNING: feature_config.json features don't match the discovered common features.")
            print(f"  in config: {len(cfg_set)} features, in data common: {len(common)}")
        else:
            print("\nfeature_config.json matches the discovered features. Good.")

print("\nCell 1 (revised) completed. Ready for next step if assertions pass.")


Base dir (user-provided): D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected
Base dir exists. Scanning for stage2 CSVs and feature config...

File discovery tentative report (search order: base_dir -> fallback -> /mnt/data):
  X_train: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected\X_train_stage2.csv
  y_train: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected\y_train_stage2.csv
  X_val: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected\X_val_stage2.csv
  y_val: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected\y_val_stage2.csv
  X_test: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected\X_test_stage2.csv
  y_test: D:\Courses\Global Acad

In [3]:
# Cell 2 (fixed): Deterministic imputer (policy B) - train-fit, group fallback, export cleaned Xs
import os
from pathlib import Path
import pandas as pd
import numpy as np
import json
import glob

# ----------------- Config -----------------
save_to_dir = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected")
alt_save_dir = Path("/mnt/data")

print("Saving imputed files to:", save_to_dir, "and", alt_save_dir)

# Reuse loaded objects from Cell 1
try:
    Xtr = X_train.copy()
    Xv  = X_val.copy()
    Xt  = X_test.copy()
except NameError:
    raise RuntimeError("X_train/X_val/X_test not found. Run Cell 1 first to load data.")

# 1) Missing counts before imputation
def missing_report(df, name):
    total_missing = int(df.isna().sum().sum())
    cols_with_missing = int((df.isna().sum()>0).sum())
    print(f"\n{name} missing: total={total_missing}, columns_with_missing={cols_with_missing}")
    if cols_with_missing:
        per_col = df.isna().sum().sort_values(ascending=False)
        print(per_col[per_col>0].head(50).to_string())

print("MISSING BEFORE IMPUTATION:")
missing_report(Xtr, "X_train")
missing_report(Xv, "X_val")
missing_report(Xt, "X_test")

# 2) Determine group columns for fallback median (prefer Branch x Category x Exam_Type)
possible_branch_cols = ["Branch", "Branch_Name", "Branch_Code", "College_Branch", "College_Branch_target_enc", "Branch_Popularity"]
possible_category_cols = ["Category", "Category_Score", "Category_Code", "Category_Name"]
exam_col = "Exam_Type"

found_branch = next((c for c in possible_branch_cols if c in Xtr.columns), None)
found_category = next((c for c in possible_category_cols if c in Xtr.columns), None)

if found_branch and found_category and exam_col in Xtr.columns:
    group_cols = [found_branch, found_category, exam_col]
elif found_category and exam_col in Xtr.columns:
    group_cols = [found_category, exam_col]
elif exam_col in Xtr.columns:
    group_cols = [exam_col]
else:
    group_cols = []

print("\nGroup columns chosen for fallback median:", group_cols)

# 3) Compute train medians per column (numeric only)
train_medians = Xtr.median(numeric_only=True)
all_missing_cols = [c for c in Xtr.columns if pd.isna(train_medians.get(c, np.nan))]
print("\nColumns with ALL-MISSING in TRAIN (train-median is NaN):", all_missing_cols)

# 4) Precompute group medians (if group_cols available)
group_medians_df = None
if group_cols:
    # numeric columns excluding group_cols to avoid duplicate insertion on reset_index
    numeric_cols = [c for c in Xtr.select_dtypes(include=[np.number]).columns.tolist() if c not in group_cols]
    if not numeric_cols:
        print("NOTE: After excluding group cols, no numeric columns remain for group medians.")
    else:
        grp = Xtr.groupby(group_cols)[numeric_cols].median().reset_index()
        group_medians_df = grp
        print("\nComputed group medians DataFrame shape:", group_medians_df.shape)
else:
    print("\nNo group columns available; will fall back to global 0 for any all-missing columns.")

# 5) Build imputation map
impute_map = {}
for col in Xtr.columns:
    med = train_medians.get(col, np.nan)
    if not pd.isna(med):
        impute_map[col] = ("const", float(med))
    else:
        impute_map[col] = ("group", None)

# 6) Apply imputation to a DataFrame
def apply_imputation(df, df_name):
    df_out = df.copy()
    # 6a. constant fills
    const_fills = {c: v for c,(kind,v) in impute_map.items() if kind=="const"}
    if const_fills:
        df_out.fillna(value=const_fills, inplace=True)
    # 6b. group fills for columns that need them
    cols_to_group_fill = [c for c,(k,v) in impute_map.items() if k=="group" and c in df_out.columns]
    if cols_to_group_fill and group_medians_df is not None and group_cols:
        left = df_out.reset_index(drop=True)
        # Merge; suffix group medians with _grpmed to avoid clobbering
        merged = left.merge(group_medians_df, on=group_cols, how='left', suffixes=("","_grpmed"))
        for c in cols_to_group_fill:
            grp_med_col = c + "_grpmed"
            if grp_med_col in merged.columns:
                mask = merged[c].isna() & merged[grp_med_col].notna()
                if mask.any():
                    merged.loc[mask, c] = merged.loc[mask, grp_med_col]
        # After attempting group fills, any remaining NA in cols_to_group_fill -> 0
        merged.fillna({c: 0.0 for c in cols_to_group_fill}, inplace=True)
        # drop any helper grpmed columns (columns ending with _grpmed)
        cols_keep = [col for col in merged.columns if not col.endswith("_grpmed")]
        df_out = merged[cols_keep]
        # ensure original column order
        df_out = df_out[df.columns]
    else:
        # no group medians: fill remaining NAs in cols_to_group_fill with 0
        if cols_to_group_fill:
            df_out.fillna({c: 0.0 for c in cols_to_group_fill}, inplace=True)
    # Final safety net
    remaining_nas = int(df_out.isna().sum().sum())
    if remaining_nas > 0:
        print(f"WARNING: {df_name} still has {remaining_nas} missing values after imputation; filling with 0 as last resort.")
        df_out = df_out.fillna(0)
    return df_out

Xtr_imp = apply_imputation(Xtr, "X_train (imputed)")
Xv_imp  = apply_imputation(Xv,  "X_val   (imputed)")
Xt_imp  = apply_imputation(Xt,  "X_test  (imputed)")

# 7) Report missing AFTER imputation
print("\nMISSING AFTER IMPUTATION (should be zero for all if Policy B succeeded):")
missing_report(Xtr_imp, "X_train_imputed")
missing_report(Xv_imp, "X_val_imputed")
missing_report(Xt_imp, "X_test_imputed")

# 8) Report which columns were filled by group fallback
cols_group_fallback = [c for c,(k,v) in impute_map.items() if k=="group"]
print("\nColumns that required GROUP fallback because train-median was NaN:", cols_group_fallback)

# 9) Save imputed CSVs
def safe_save(df, dst_path):
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(dst_path, index=False)
    print("WROTE:", dst_path)

base = Path(save_to_dir)
out_files = {
    "X_train_imputed": base / (Path(found["X_train"]).stem + "_imputed_stage3.csv"),
    "X_val_imputed":   base / (Path(found["X_val"]).stem + "_imputed_stage3.csv"),
    "X_test_imputed":  base / (Path(found["X_test"]).stem + "_imputed_stage3.csv"),
}
safe_save(Xtr_imp, out_files["X_train_imputed"])
safe_save(Xv_imp,  out_files["X_val_imputed"])
safe_save(Xt_imp,  out_files["X_test_imputed"])

# try saving to /mnt/data as well
try:
    if alt_save_dir.exists():
        safe_save(Xtr_imp, Path(alt_save_dir)/out_files["X_train_imputed"].name)
        safe_save(Xv_imp,  Path(alt_save_dir)/out_files["X_val_imputed"].name)
        safe_save(Xt_imp,  Path(alt_save_dir)/out_files["X_test_imputed"].name)
except Exception as e:
    print("Could not write to /mnt/data:", e)

# 10) Export imputation map
imp_map_path = base / "imputation_map_stage3.json"
with open(imp_map_path, "w") as f:
    json.dump({"impute_map": {k:v for k,v in impute_map.items()}, "group_cols": group_cols}, f, indent=2)
print("WROTE imputation map to:", imp_map_path)

# expose imputed DFs to globals
X_train_imputed = Xtr_imp
X_val_imputed = Xv_imp
X_test_imputed = Xt_imp

print("\nCell 2 (fixed) completed. Objects available: X_train_imputed, X_val_imputed, X_test_imputed")


Saving imputed files to: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected and \mnt\data
MISSING BEFORE IMPUTATION:

X_train missing: total=0, columns_with_missing=0

X_val missing: total=0, columns_with_missing=0

X_test missing: total=0, columns_with_missing=0

Group columns chosen for fallback median: ['College_Branch_target_enc', 'Category_Score', 'Exam_Type']

Columns with ALL-MISSING in TRAIN (train-median is NaN): []

Computed group medians DataFrame shape: (23471, 32)

MISSING AFTER IMPUTATION (should be zero for all if Policy B succeeded):

X_train_imputed missing: total=0, columns_with_missing=0

X_val_imputed missing: total=0, columns_with_missing=0

X_test_imputed missing: total=0, columns_with_missing=0

Columns that required GROUP fallback because train-median was NaN: []
WROTE: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected\X_train_stage2_imputed_stage

In [4]:
# Cell 3 (fixed): Baselines — Naive lag-1 within groups, Linear Regression, Ridge
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reuse imputed DFs and targets
Xtr = X_train_imputed.copy()
Xv  = X_val_imputed.copy()
Xt  = X_test_imputed.copy()

ytr = y_train.copy()
yv  = y_val.copy()
yt  = y_test.copy()

# Locked features
try:
    locked_features
except NameError:
    locked_features = list(Xtr.columns)

# 1) Naive lag-1 baseline
group_keys = ["College_Code_target_enc", "College_Branch_target_enc", "Category_Score", "Exam_Type"]
print("Group keys used for naive lag-1:", group_keys)

df_tr = Xtr[ group_keys + ["Year"] ].copy().reset_index(drop=True)
df_tr["y"] = ytr.values

df_tr["group_tuple"] = list(df_tr[group_keys].itertuples(index=False, name=None))
group_year_mean = df_tr.groupby(["group_tuple","Year"])["y"].mean().to_dict()
group_mean = df_tr.groupby(["group_tuple"])["y"].mean().to_dict()
global_mean = float(df_tr["y"].mean())

def naive_lag1_predict(X_df):
    preds = []
    years = X_df["Year"].astype(int).values
    groups = list(X_df[group_keys].itertuples(index=False, name=None))
    for g, yr in zip(groups, years):
        key1 = (g, int(yr-1))
        if key1 in group_year_mean:
            preds.append(group_year_mean[key1])
            continue
        if g in group_mean:
            preds.append(group_mean[g])
            continue
        preds.append(global_mean)
    return np.array(preds)

naive_val_pred = naive_lag1_predict(Xv)
naive_test_pred = naive_lag1_predict(Xt)

def report_preds(true, pred, name):
    mae = mean_absolute_error(true, pred)
    rmse = float(np.sqrt(mean_squared_error(true, pred)))
    r2 = r2_score(true, pred)
    print(f"\n{name} — MAE: {mae:.4f}, RMSE: {rmse:.4f}, R^2: {r2:.4f}")
    return mae, rmse, r2

print("\nNAIVE LAG-1 BASELINE RESULTS:")
naive_val_metrics = report_preds(yv, naive_val_pred, "Validation")
naive_test_metrics = report_preds(yt, naive_test_pred, "Test")

# 2) Linear Regression and Ridge (numeric features only)
Xtr_num = Xtr[locked_features].select_dtypes(include=[np.number])
Xv_num = Xv[locked_features].select_dtypes(include=[np.number])
Xt_num = Xt[locked_features].select_dtypes(include=[np.number])

cols_num = list(Xtr_num.columns)
Xv_num = Xv_num[cols_num]
Xt_num = Xt_num[cols_num]

print("\nNumeric features used for linear models:", len(cols_num))

def fit_and_eval(model, X_train, y_train, X_val, y_val, X_test, y_test, name):
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)
    test_pred = pipe.predict(X_test)
    print(f"\n{name} results:")
    print("  Val:", end=" ")
    report_preds(y_val, val_pred, "Validation")
    print("  Test:", end=" ")
    report_preds(y_test, test_pred, "Test")
    return pipe, val_pred, test_pred

# Linear Regression
lr_model = LinearRegression()
lr_pipe, lr_val_pred, lr_test_pred = fit_and_eval(lr_model, Xtr_num, ytr, Xv_num, yv, Xt_num, yt, "LinearRegression")

# Ridge
ridge_model = Ridge(alpha=1.0)
ridge_pipe, ridge_val_pred, ridge_test_pred = fit_and_eval(ridge_model, Xtr_num, ytr, Xv_num, yv, Xt_num, yt, "Ridge(alpha=1.0)")

# Summary table
rows = []
rows.append(["NaiveLag1", *naive_val_metrics, *naive_test_metrics])
rows.append(["LinearRegression", float(mean_absolute_error(yv, lr_val_pred)), float(np.sqrt(mean_squared_error(yv, lr_val_pred))), float(r2_score(yv, lr_val_pred)), float(mean_absolute_error(yt, lr_test_pred)), float(np.sqrt(mean_squared_error(yt, lr_test_pred))), float(r2_score(yt, lr_test_pred))])
rows.append(["Ridge_alpha1.0", float(mean_absolute_error(yv, ridge_val_pred)), float(np.sqrt(mean_squared_error(yv, ridge_val_pred))), float(r2_score(yv, ridge_val_pred)), float(mean_absolute_error(yt, ridge_test_pred)), float(np.sqrt(mean_squared_error(yt, ridge_test_pred))), float(r2_score(yt, ridge_test_pred))])
summary = pd.DataFrame(rows, columns=["Model","Val_MAE","Val_RMSE","Val_R2","Test_MAE","Test_RMSE","Test_R2"])
print("\nBASELINE SUMMARY:")
print(summary.round(6).to_string(index=False))


Group keys used for naive lag-1: ['College_Code_target_enc', 'College_Branch_target_enc', 'Category_Score', 'Exam_Type']

NAIVE LAG-1 BASELINE RESULTS:

Validation — MAE: 41955.4182, RMSE: 53105.2513, R^2: -0.0565

Test — MAE: 60419.4921, RMSE: 78861.0893, R^2: -0.3531

Numeric features used for linear models: 32

LinearRegression results:
  Val: 
Validation — MAE: 22405.9970, RMSE: 30136.0709, R^2: 0.6598
  Test: 
Test — MAE: 31201.1439, RMSE: 44401.7992, R^2: 0.5711

Ridge(alpha=1.0) results:
  Val: 
Validation — MAE: 22405.5750, RMSE: 30135.8478, R^2: 0.6598
  Test: 
Test — MAE: 31201.2589, RMSE: 44402.6052, R^2: 0.5710

BASELINE SUMMARY:
           Model      Val_MAE     Val_RMSE    Val_R2     Test_MAE    Test_RMSE   Test_R2
       NaiveLag1 41955.418182 53105.251342 -0.056543 60419.492069 78861.089336 -0.353095
LinearRegression 22405.997048 30136.070889  0.659760 31201.143895 44401.799244  0.571053
  Ridge_alpha1.0 22405.574952 30135.847834  0.659765 31201.258890 44402.605169  0.5

In [5]:
# Cell 4 (robust, full): Train XGBoost via xgb.train with version-safe predict & saving
import os, json, time, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ----------------- Config -----------------
save_to_dir = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected")
alt_save_dir = Path("/mnt/data")
save_to_dir.mkdir(parents=True, exist_ok=True)

# Reuse imputed datasets and targets from prior cells
try:
    Xtr_all = X_train_imputed.copy()
    Xv_all  = X_val_imputed.copy()
    Xt_all  = X_test_imputed.copy()
    ytr_all = y_train.copy()
    yv_all  = y_val.copy()
    yt_all  = y_test.copy()
except NameError as e:
    raise RuntimeError("Required dataframes/targets not found in notebook state. Run Cell 1 and Cell 2 first.") from e

# Features
try:
    features = locked_features
except NameError:
    features = list(Xtr_all.columns)

print(f"Training using {len(features)} features. Example: {features[:6]}")

# Build DMatrices
dtrain = xgb.DMatrix(Xtr_all[features], label=ytr_all)
dval   = xgb.DMatrix(Xv_all[features], label=yv_all)
dtest  = xgb.DMatrix(Xt_all[features], label=yt_all)

# Params (CPU)
params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.03,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "alpha": 1.0,    # L1
    "lambda": 2.0,   # L2
    "min_child_weight": 50,
    "verbosity": 1,
    "seed": 42,
    "tree_method": "hist"
}
num_boost_round = 5000
early_stopping_rounds = 300

print("\nXGBoost params (summary):")
for k,v in params.items():
    print(f"  {k}: {v}")
print(f"num_boost_round={num_boost_round}, early_stopping_rounds={early_stopping_rounds}")

# Train
evals = [(dtrain, "train"), (dval, "val")]
evals_result = {}
t0 = time.time()
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=evals,
    early_stopping_rounds=early_stopping_rounds,
    evals_result=evals_result,
    verbose_eval=50
)
t1 = time.time()
print(f"\nTraining done in {(t1-t0)/60:.2f} minutes")

# Best iteration robust fetch
best_iter = None
if hasattr(bst, "best_iteration") and bst.best_iteration is not None:
    best_iter = int(bst.best_iteration)
elif hasattr(bst, "best_ntree_limit") and bst.best_ntree_limit is not None:
    # older versions
    try:
        best_iter = int(bst.best_ntree_limit) - 1
    except Exception:
        best_iter = None

print("Detected best iteration:", best_iter)

# Version-safe predict helper
def safe_predict(booster, dmatrix, best_iter):
    # prefer iteration_range
    try:
        if best_iter is not None:
            pred = booster.predict(dmatrix, iteration_range=(0, best_iter+1))
        else:
            pred = booster.predict(dmatrix)
        return pred
    except TypeError:
        # try legacy ntree_limit
        try:
            if best_iter is not None:
                pred = booster.predict(dmatrix, ntree_limit=best_iter+1)
            else:
                pred = booster.predict(dmatrix)
            return pred
        except Exception:
            # final fallback
            return booster.predict(dmatrix)

# Predict
y_val_pred = safe_predict(bst, dval, best_iter)
y_test_pred = safe_predict(bst, dtest, best_iter)

# Metrics
def metrics(true, pred):
    mae = mean_absolute_error(true, pred)
    rmse = float(np.sqrt(mean_squared_error(true, pred)))
    r2 = r2_score(true, pred)
    return mae, rmse, r2

val_mae, val_rmse, val_r2 = metrics(yv_all, y_val_pred)
test_mae, test_rmse, test_r2 = metrics(yt_all, y_test_pred)

print("\nFINAL METRICS:")
print(f"  Validation (2023) — MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f}, R^2: {val_r2:.4f}")
print(f"  Test (2024)       — MAE: {test_mae:.4f}, RMSE: {test_rmse:.4f}, R^2: {test_r2:.4f}")

# Save booster and wrapper
booster_path = save_to_dir / f"xgb_booster_stage3_bestiter{best_iter if best_iter is not None else 'none'}.json"
bst.save_model(str(booster_path))
print("Saved booster to:", booster_path)

wrapper = {"booster_path": str(booster_path), "features": features, "params": params, "best_iteration": best_iter}
joblib_path = save_to_dir / f"xgb_wrapper_stage3_bestiter{best_iter if best_iter is not None else 'none'}.joblib"
joblib.dump(wrapper, joblib_path)
print("Saved wrapper to:", joblib_path)

meta = {
    "val_mae": float(val_mae), "val_rmse": float(val_rmse), "val_r2": float(val_r2),
    "test_mae": float(test_mae), "test_rmse": float(test_rmse), "test_r2": float(test_r2),
    "best_iteration": best_iter, "num_boost_round": num_boost_round,
    "early_stopping_rounds": early_stopping_rounds, "params": params
}
meta_path = save_to_dir / "xgb_train_meta_stage3.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print("Saved training meta to:", meta_path)

# Save predictions
np.save(save_to_dir / "y_val_pred_xgb_stage3.npy", y_val_pred)
np.save(save_to_dir / "y_test_pred_xgb_stage3.npy", y_test_pred)
print("Saved predictions to:", save_to_dir)

# attempt copy to /mnt/data
try:
    if alt_save_dir.exists():
        bst.save_model(str(Path(alt_save_dir)/booster_path.name))
        joblib.dump(wrapper, Path(alt_save_dir)/joblib_path.name)
        with open(Path(alt_save_dir)/meta_path.name, "w") as f:
            json.dump(meta, f, indent=2)
        np.save(Path(alt_save_dir)/"y_val_pred_xgb_stage3.npy", y_val_pred)
        np.save(Path(alt_save_dir)/"y_test_pred_xgb_stage3.npy", y_test_pred)
        print("Also saved copies to /mnt/data")
except Exception as e:
    print("Could not copy to /mnt/data:", e)

# expose variables for next steps
booster = bst
y_val_pred_xgb = y_val_pred
y_test_pred_xgb = y_test_pred
evals_result_dict = evals_result

print("\nCell complete. Objects available: booster, y_val_pred_xgb, y_test_pred_xgb, evals_result_dict")


Training using 32 features. Example: ['Branch_Popularity', 'Category_Score', 'College_Branch_target_enc', 'College_Code_target_enc', 'College_Tier_Numeric', 'Exam_Type']

XGBoost params (summary):
  objective: reg:squarederror
  learning_rate: 0.03
  max_depth: 8
  subsample: 0.8
  colsample_bytree: 0.8
  alpha: 1.0
  lambda: 2.0
  min_child_weight: 50
  verbosity: 1
  seed: 42
  tree_method: hist
num_boost_round=5000, early_stopping_rounds=300
[0]	train-rmse:44197.96454	val-rmse:52071.77365
[50]	train-rmse:22461.07018	val-rmse:32597.43210
[100]	train-rmse:19593.22837	val-rmse:29253.75320
[150]	train-rmse:18842.74102	val-rmse:28247.18628
[200]	train-rmse:18455.01689	val-rmse:27865.69941
[250]	train-rmse:18177.54331	val-rmse:27717.75182
[300]	train-rmse:17963.78991	val-rmse:27660.62325
[350]	train-rmse:17768.37202	val-rmse:27542.05968
[400]	train-rmse:17616.65707	val-rmse:27445.70080
[450]	train-rmse:17479.22007	val-rmse:27373.59803
[500]	train-rmse:17334.06213	val-rmse:27308.09568
[550

In [6]:
# Cell: Move Stage 3 XGBoost artifacts to proper model directory

import shutil
from pathlib import Path

# -------- CONFIG --------
# Adjust this ONLY if your notebook is not running at repo root.
PROJECT_ROOT = Path(".")  # assumes CWD is the repo root

DATA_DIR = PROJECT_ROOT / "kcet_ml_project" / "data" / "stage2_v2_corrected"
DEST_DIR = PROJECT_ROOT / "kcet_ml_project" / "models" / "xgboost_stage3"

DEST_DIR.mkdir(parents=True, exist_ok=True)

# File patterns to relocate
patterns = [
    "xgb_booster_stage3_*.json",
    "xgb_wrapper_stage3_*.joblib",
    "xgb_train_meta_stage3.json",
    "y_val_pred_xgb_stage3.npy",
    "y_test_pred_xgb_stage3.npy",
]

moved_files = []

for pattern in patterns:
    for src_file in DATA_DIR.glob(pattern):
        dst_file = DEST_DIR / src_file.name
        shutil.move(str(src_file), str(dst_file))
        moved_files.append(dst_file)

if not moved_files:
    print("⚠️ No Stage 3 XGBoost artifacts found in:", DATA_DIR)
else:
    print("✅ Moved the following files to:", DEST_DIR)
    for f in moved_files:
        print("  -", f.name)


✅ Moved the following files to: kcet_ml_project\models\xgboost_stage3
  - xgb_booster_stage3_bestiter1078.json
  - xgb_wrapper_stage3_bestiter1078.joblib
  - xgb_train_meta_stage3.json
  - y_val_pred_xgb_stage3.npy
  - y_test_pred_xgb_stage3.npy


In [7]:
# Cell: Leakage sanity checks (Val↔Test gap, label-shuffle, drop intra-year round features)
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path
import joblib
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils import shuffle

# ----------------- Config -----------------
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "kcet_ml_project" / "data"
MODEL_DIR = PROJECT_ROOT / "kcet_ml_project" / "models" / "xgboost_stage3"
# fallback to previous save_to_dir if available
if not MODEL_DIR.exists():
    MODEL_DIR = Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected")

print("Model dir:", MODEL_DIR)

# Load original data & features (imputed)
try:
    Xtr = X_train_imputed.copy()
    Xv  = X_val_imputed.copy()
    Xt  = X_test_imputed.copy()
    ytr = y_train.copy()
    yv  = y_val.copy()
    yt  = y_test.copy()
except NameError:
    raise RuntimeError("Imputed datasets or targets not found in notebook state. Run previous cells to load them.")

# Load params and wrapper if present, else infer reasonable defaults
params = None
features = None
meta_path = MODEL_DIR / "xgb_train_meta_stage3.json"
wrapper_candidates = list(MODEL_DIR.glob("xgb_wrapper_stage3*.joblib"))
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    params = meta.get("params", None)
    saved_val_mae = meta.get("val_mae", None)
    saved_test_mae = meta.get("test_mae", None)
else:
    meta = {}
if wrapper_candidates and features is None:
    try:
        wrapper = joblib.load(wrapper_candidates[0])
        features = wrapper.get("features", None)
        params = wrapper.get("params", params)
    except Exception:
        features = None

# Fallbacks
if features is None:
    features = sorted(list(Xtr.columns))
if params is None:
    # minimal params mirroring your earlier run
    params = {
        "objective": "reg:squarederror",
        "learning_rate": 0.03,
        "max_depth": 8,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "alpha": 1.0,
        "lambda": 2.0,
        "min_child_weight": 50,
        "verbosity": 1,
        "seed": 42,
        "tree_method": "hist"
    }

print("\nUsing params summary:")
print({k: params[k] for k in ["learning_rate","max_depth","subsample","colsample_bytree","alpha","lambda","min_child_weight"] if k in params})
print("Number of features used:", len(features))

# helper metrics
def metrics(true, pred):
    mae = mean_absolute_error(true, pred)
    rmse = float(np.sqrt(mean_squared_error(true, pred)))
    r2 = r2_score(true, pred)
    return mae, rmse, r2

# 1) Val↔Test gap using saved preds if available, else compute using current booster if present
print("\n1) Val vs Test gap check")
val_pred_path = MODEL_DIR / "y_val_pred_xgb_stage3.npy"
test_pred_path = MODEL_DIR / "y_test_pred_xgb_stage3.npy"
if val_pred_path.exists() and test_pred_path.exists():
    y_val_pred = np.load(val_pred_path)
    y_test_pred = np.load(test_pred_path)
    print("Loaded saved predictions from model dir.")
else:
    # if booster exists in memory or saved, try to use it
    y_val_pred = None
    y_test_pred = None
    # attempt to load booster
    booster_files = list(MODEL_DIR.glob("xgb_booster_stage3*.json"))
    if booster_files:
        try:
            bst = xgb.Booster()
            bst.load_model(str(booster_files[0]))
            # build DMatrices and best_iteration safe retrieval
            dval = xgb.DMatrix(Xv[features], label=yv)
            dtest = xgb.DMatrix(Xt[features], label=yt)
            best_iter = getattr(bst, "best_iteration", None)
            try:
                if best_iter is not None:
                    y_val_pred = bst.predict(dval, iteration_range=(0, int(best_iter)+1))
                    y_test_pred = bst.predict(dtest, iteration_range=(0, int(best_iter)+1))
                else:
                    y_val_pred = bst.predict(dval)
                    y_test_pred = bst.predict(dtest)
                print("Predictions computed from saved booster.")
            except TypeError:
                # older xgboost
                if best_iter is not None:
                    y_val_pred = bst.predict(dval, ntree_limit=int(best_iter)+1)
                    y_test_pred = bst.predict(dtest, ntree_limit=int(best_iter)+1)
                else:
                    y_val_pred = bst.predict(dval)
                    y_test_pred = bst.predict(dtest)
                print("Predictions computed from saved booster (legacy predict).")
        except Exception as e:
            print("Could not load booster for preds:", e)
    if y_val_pred is None:
        raise RuntimeError("No predictions available (no saved preds and could not compute from booster). Please run XGBoost training cell first.")

val_mae, val_rmse, val_r2 = metrics(yv, y_val_pred)
test_mae, test_rmse, test_r2 = metrics(yt, y_test_pred)
ratio = test_mae / val_mae if val_mae > 0 else float("inf")
print(f"  Val MAE: {val_mae:.2f}, Test MAE: {test_mae:.2f}, Ratio Test/Val: {ratio:.3f}")
if ratio > 3:
    print("  >>> FAIL: Test/Val MAE ratio > 3.0 — possible leakage or dataset mismatch. Investigate.")
elif ratio > 2.5:
    print("  >>> WARNING: Test/Val MAE ratio > 2.5 — large generalisation gap. Proceed cautiously.")
else:
    print("  >>> PASS: Test/Val MAE ratio within acceptable bounds.")

# 2) Label-shuffle test
print("\n2) Label-shuffle test (train on shuffled y). Expect Val R^2 <= 0")
# Create shuffled y for training only (same index)
ytr_shuffled = ytr.sample(frac=1.0, random_state=12345).reset_index(drop=True)
# Important: align Xtr rows with shuffled labels (we shuffle the labels independently)
# We'll rebuild DMatrix with the shuffled labels
dtrain_shuf = xgb.DMatrix(Xtr[features].values, label=ytr_shuffled.values)
dval_dm = xgb.DMatrix(Xv[features].values, label=yv.values)

# Train a smaller budget (we'll use early stopping to limit waste)
num_boost_round = 2000
early_stopping_rounds = 100
evals = [(dtrain_shuf, "train_shuf"), (dval_dm, "val")]

evals_result_shuf = {}
t0 = time.time()
bst_shuf = xgb.train(
    params,
    dtrain_shuf,
    num_boost_round=num_boost_round,
    evals=evals,
    early_stopping_rounds=early_stopping_rounds,
    evals_result=evals_result_shuf,
    verbose_eval=100
)
t1 = time.time()
print(f"  Trained on shuffled labels in {(t1-t0):.1f}s. Best iteration: {getattr(bst_shuf,'best_iteration',None)}")

# Predict and measure R2 on Val
try:
    best_iter_shuf = getattr(bst_shuf, "best_iteration", None)
    if best_iter_shuf is not None:
        yv_shuf_pred = bst_shuf.predict(dval_dm, iteration_range=(0, int(best_iter_shuf)+1))
    else:
        yv_shuf_pred = bst_shuf.predict(dval_dm)
except TypeError:
    if best_iter_shuf is not None:
        yv_shuf_pred = bst_shuf.predict(dval_dm, ntree_limit=int(best_iter_shuf)+1)
    else:
        yv_shuf_pred = bst_shuf.predict(dval_dm)

mae_shuf, rmse_shuf, r2_shuf = metrics(yv, yv_shuf_pred)
print(f"  Shuffled Val — MAE: {mae_shuf:.2f}, RMSE: {rmse_shuf:.2f}, R^2: {r2_shuf:.4f}")
if r2_shuf <= 0:
    print("  >>> PASS: R^2 <= 0 on Val with shuffled labels.")
else:
    print("  >>> FAIL: R^2 > 0 with shuffled labels — potential leakage or target info in features.")

# 3) Feature drop test: drop intra-year round features and retrain
print("\n3) Feature-drop test (drop intra-year / round features and compare)")

# Heuristic to detect intra-year/round features
drop_patterns = ["_L1R", "_L2R", "round", "Round", "n_rounds_hist", "is_first_round", "L1R", "L2R"]
cols_to_drop = []
for c in features:
    if any(pat.lower() in c.lower() for pat in drop_patterns):
        cols_to_drop.append(c)
cols_to_drop = sorted(set(cols_to_drop))
print("Detected candidate intra-year/round features to drop:", cols_to_drop)

if not cols_to_drop:
    print("  No round-like features detected. Skipping feature-drop test.")
else:
    features_reduced = [c for c in features if c not in cols_to_drop]
    print(f"  Running XGBoost on reduced feature set ({len(features_reduced)} features)")

    # Build DMatrices
    dtrain_red = xgb.DMatrix(Xtr[features_reduced].values, label=ytr.values)
    dval_red = xgb.DMatrix(Xv[features_reduced].values, label=yv.values)
    dtest_red = xgb.DMatrix(Xt[features_reduced].values, label=yt.values)

    evals_result_red = {}
    t0 = time.time()
    bst_red = xgb.train(
        params,
        dtrain_red,
        num_boost_round=2000,
        evals=[(dtrain_red, "train_red"), (dval_red, "val_red")],
        early_stopping_rounds=100,
        evals_result=evals_result_red,
        verbose_eval=100
    )
    t1 = time.time()
    print(f"  Retrained reduced model in {(t1-t0):.1f}s. Best iter: {getattr(bst_red,'best_iteration',None)}")

    # Predict & compare
    try:
        bi_red = getattr(bst_red, "best_iteration", None)
        if bi_red is not None:
            yv_red = bst_red.predict(dval_red, iteration_range=(0, int(bi_red)+1))
            yt_red = bst_red.predict(dtest_red, iteration_range=(0, int(bi_red)+1))
        else:
            yv_red = bst_red.predict(dval_red)
            yt_red = bst_red.predict(dtest_red)
    except TypeError:
        bi_red = getattr(bst_red, "best_iteration", None)
        if bi_red is not None:
            yv_red = bst_red.predict(dval_red, ntree_limit=int(bi_red)+1)
            yt_red = bst_red.predict(dtest_red, ntree_limit=int(bi_red)+1)
        else:
            yv_red = bst_red.predict(dval_red)
            yt_red = bst_red.predict(dtest_red)

    val_mae_red, val_rmse_red, val_r2_red = metrics(yv, yv_red)
    test_mae_red, test_rmse_red, test_r2_red = metrics(yt, yt_red)

    print(f"\n  Reduced model Val MAE: {val_mae_red:.2f}, Test MAE: {test_mae_red:.2f}")
    if val_mae_red < val_mae:
        print("  >>> WARNING/FLAG: Reduced model (no round-features) has LOWER Val MAE than original. This suggests possible overfitting to round features — audit them carefully.")
    else:
        print("  >>> PASS: Removing round-like features did not improve Val MAE (expected).")
    print("\nSummary comparison:")
    print(f" Original Val MAE: {val_mae:.2f}, Reduced Val MAE: {val_mae_red:.2f}")
    print(f" Original Test MAE: {test_mae:.2f}, Reduced Test MAE: {test_mae_red:.2f}")

print("\nLeakage checks complete.")


Model dir: kcet_ml_project\models\xgboost_stage3

Using params summary:
{'learning_rate': 0.03, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.8, 'alpha': 1.0, 'lambda': 2.0, 'min_child_weight': 50}
Number of features used: 32

1) Val vs Test gap check
Loaded saved predictions from model dir.
  Val MAE: 18329.14, Test MAE: 32039.35, Ratio Test/Val: 1.748
  >>> PASS: Test/Val MAE ratio within acceptable bounds.

2) Label-shuffle test (train on shuffled y). Expect Val R^2 <= 0
[0]	train_shuf-rmse:45181.73135	val-rmse:53109.84783
[100]	train_shuf-rmse:44827.23085	val-rmse:52947.73856
[200]	train_shuf-rmse:44540.67784	val-rmse:52839.50056
[300]	train_shuf-rmse:44257.49174	val-rmse:52829.69017
[336]	train_shuf-rmse:44167.47570	val-rmse:52802.58328
  Trained on shuffled labels in 5.1s. Best iteration: 237
  Shuffled Val — MAE: 41704.38, RMSE: 52783.37, R^2: -0.0438
  >>> PASS: R^2 <= 0 on Val with shuffled labels.

3) Feature-drop test (drop intra-year / round features and compare)


In [8]:
# Cell 6: Expanding Time-Aware Cross-Validation (2020→2021, 2020–21→2022)

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

# Ensure features & data exist
try:
    features = locked_features
    Xtr_all = X_train_imputed.copy()
    ytr_all = y_train.copy()
except Exception as e:
    raise RuntimeError("Missing X_train_imputed or locked_features. Run previous cells.") from e

# Split by Year
df_tr = Xtr_all.copy()
df_tr["y"] = ytr_all.values

# Folds
folds = [
    {"train_years": [2020],       "val_year": 2021},
    {"train_years": [2020, 2021], "val_year": 2022},
]

# Params (same as your Stage 3 run)
params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.03,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "alpha": 1.0,
    "lambda": 2.0,
    "min_child_weight": 50,
    "verbosity": 1,
    "seed": 42,
    "tree_method": "hist"
}

num_boost_round = 3000
early_stopping_rounds = 200

results = []

print("\n===== EXPANDING TIME-AWARE CV =====")

for i, fold in enumerate(folds, 1):
    train_years = fold["train_years"]
    val_year = fold["val_year"]
    
    print(f"\n--- Fold {i}: Train {train_years} -> Validate {val_year} ---")
    
    X_train = df_tr[df_tr["Year"].isin(train_years)][features].values
    y_train = df_tr[df_tr["Year"].isin(train_years)]["y"].values
    X_val   = df_tr[df_tr["Year"] == val_year][features].values
    y_val   = df_tr[df_tr["Year"] == val_year]["y"].values
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval   = xgb.DMatrix(X_val, label=y_val)
    
    evals_result = {}
    t0 = time.time()
    
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=early_stopping_rounds,
        evals_result=evals_result,
        verbose_eval=100,
    )
    
    t1 = time.time()
    best_iter = getattr(model, "best_iteration", None)
    print(f"Best iteration: {best_iter}, Time: {(t1-t0):.1f}s")
    
    # Safe prediction
    try:
        preds = model.predict(dval, iteration_range=(0, best_iter+1))
    except:
        preds = model.predict(dval)
    
    mae = mean_absolute_error(y_val, preds)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)
    
    print(f"Fold {i} MAE:  {mae:.2f}")
    print(f"Fold {i} RMSE: {rmse:.2f}")
    print(f"Fold {i} R²:   {r2:.4f}")
    
    results.append((mae, rmse, r2))

# Summary
maes  = [r[0] for r in results]
rmses = [r[1] for r in results]
r2s   = [r[2] for r in results]

print("\n===== CV SUMMARY =====")
print(f"MAE:  mean={np.mean(maes):.2f}, std={np.std(maes):.2f}")
print(f"RMSE: mean={np.mean(rmses):.2f}, std={np.std(rmses):.2f}")
print(f"R²:   mean={np.mean(r2s):.4f}, std={np.std(r2s):.4f}")

print("\nCompare with your Val (2023):")
print(f"Val2023 MAE = {18329.14:.2f}")

if np.mean(maes) > 25000:
    print("⚠️ CV MAE is high → instability across years.")
elif np.std(maes) > 4000:
    print("⚠️ High variance → temporal drift present.")
else:
    print("✅ CV performance stable. Safe to proceed to tuning.")



===== EXPANDING TIME-AWARE CV =====

--- Fold 1: Train [2020] -> Validate 2021 ---
[0]	train-rmse:41654.97221	val-rmse:47835.16512
[100]	train-rmse:19121.10686	val-rmse:28424.64040
[200]	train-rmse:18129.19807	val-rmse:28943.22459
[277]	train-rmse:17722.27811	val-rmse:29166.66394
Best iteration: 78, Time: 2.0s
Fold 1 MAE:  21956.05
Fold 1 RMSE: 28289.28
Fold 1 R²:   0.6507

--- Fold 2: Train [2020, 2021] -> Validate 2022 ---
[0]	train-rmse:44988.03627	val-rmse:42923.76182
[100]	train-rmse:20419.82978	val-rmse:21419.27152
[200]	train-rmse:19345.23169	val-rmse:21580.86088
[300]	train-rmse:18852.52338	val-rmse:21800.49990
[313]	train-rmse:18803.23238	val-rmse:21830.17884
Best iteration: 114, Time: 3.8s
Fold 2 MAE:  14719.22
Fold 2 RMSE: 21386.49
Fold 2 R²:   0.7615

===== CV SUMMARY =====
MAE:  mean=18337.64, std=3618.41
RMSE: mean=24837.88, std=3451.40
R²:   mean=0.7061, std=0.0554

Compare with your Val (2023):
Val2023 MAE = 18329.14
✅ CV performance stable. Safe to proceed to tuning.


In [12]:
# Cell: Robust Optuna tuning runner that auto-finds correct y to match X_train_imputed
import os, sys, json, time, math
from pathlib import Path
import numpy as np
import pandas as pd

# try to import optuna, install if missing
try:
    import optuna
except Exception:
    print("Optuna not found — attempting to install optuna (this may take a moment)...")
    !{sys.executable} -m pip install -q optuna
    import optuna

import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------- Config ----------
PROJECT_ROOT = Path(".")
MODEL_DIR = PROJECT_ROOT / "kcet_ml_project" / "models" / "xgboost_stage3"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRIALS = 40  # change if you want to run more
RANDOM_SEED = 42

# ensure X_train_imputed exists
try:
    X_all = X_train_imputed.copy()
except NameError:
    raise RuntimeError("X_train_imputed not found. Run the earlier cells (Cell 2) that produce X_train_imputed.")

nX = len(X_all)
print("Rows in X_train_imputed (nX):", nX)

# Helper: try load candidate y files from a list of dirs
candidate_dirs = [
    PROJECT_ROOT / "kcet_ml_project" / "data",
    PROJECT_ROOT / "kcet_ml_project" / "data" / "stage2_v2_corrected",
    Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected"),
    Path("/mnt/data"),
    Path("."),
]

candidate_patterns = ["y_train*.csv", "*y_train*.csv", "y_*.csv", "train_*y*.csv", "*_y_*.csv", "train_stage2_final.csv", "val_stage2_final.csv"]

found_y = None
found_y_path = None

checked_paths = set()
for d in candidate_dirs:
    if not d.exists():
        continue
    for pat in candidate_patterns:
        for p in d.glob(pat):
            p = p.resolve()
            if p in checked_paths:
                continue
            checked_paths.add(p)
            try:
                tmp = pd.read_csv(p, header=0)
                # if file has more than 1 col, try to find a single numeric column as y
                if tmp.shape[1] == 1:
                    arr = tmp.iloc[:,0].values
                else:
                    # prefer columns named exactly like 'y','target','cutoff','label'
                    col_candidates = [c for c in tmp.columns if c.lower() in ("y","target","cutoff","label","score")]
                    if col_candidates:
                        arr = tmp[col_candidates[0]].values
                    else:
                        # fallback: if there is a numeric column and length equals nX, use first numeric
                        numeric_cols = tmp.select_dtypes(include=[np.number]).columns.tolist()
                        if numeric_cols:
                            arr = tmp[numeric_cols[0]].values
                        else:
                            continue
                if len(arr) == nX:
                    found_y = arr
                    found_y_path = p
                    break
            except Exception:
                continue
        if found_y is not None:
            break
    if found_y is not None:
        break

# If not found on disk, check in-memory y_train/y_val/y_test objects
if found_y is None:
    mem_candidates = {}
    for name in ("y_train", "y_val", "y_test", "y_all", "ytr_all"):
        if name in globals():
            try:
                arr = np.asarray(globals()[name])
                mem_candidates[name] = arr
            except Exception:
                pass
    # check for exact length match
    for name, arr in mem_candidates.items():
        if len(arr) == nX:
            found_y = arr
            found_y_path = f"in-memory:{name}"
            break

# If still none, try concatenation of y_train+y_val+y_test
if found_y is None:
    concat_possible = []
    for name in ("y_train","y_val","y_test"):
        if name in globals():
            concat_possible.append(np.asarray(globals()[name]))
    if concat_possible and sum(len(a) for a in concat_possible) == nX:
        found_y = np.concatenate(concat_possible)
        found_y_path = "concatenated:y_train+y_val+y_test"

if found_y is None:
    # As a last resort: try to load X_train_stage2.csv (if it contains label column)
    probe_files = []
    for d in candidate_dirs:
        if not d.exists(): continue
        probe_files.extend(list(d.glob("X_train_stage2*.csv")))
        probe_files.extend(list(d.glob("*train_stage2*.csv")))
    for p in probe_files:
        try:
            tmp = pd.read_csv(p)
            # try to find a label-like column
            label_cols = [c for c in tmp.columns if c.lower() in ("y","target","cutoff","label","score")]
            if label_cols:
                arr = tmp[label_cols[0]].values
                if len(arr) == nX:
                    found_y = arr
                    found_y_path = p
                    break
        except Exception:
            pass

if found_y is None:
    raise RuntimeError(
        "Could not automatically find a y vector matching X_train_imputed length (nX).\n"
        "I searched common data folders and in-memory variables. Please ensure the correct y_train CSV is in one of these locations or that y_train in memory matches X_train_imputed rows."
    )

print("Using y from:", found_y_path, " (length =", len(found_y), ")")

# Now we have X_all (X_train_imputed) and found_y
# Continue with Optuna tuning (Fold 2: train 2020-2021 -> val 2022)
features = locked_features
df_tr = X_all.copy()
df_tr["y"] = found_y

train_mask = df_tr["Year"].isin([2020, 2021])
val_mask = df_tr["Year"] == 2022

X_train_tune = df_tr[train_mask][features]
y_train_tune = df_tr[train_mask]["y"]
X_val_tune = df_tr[val_mask][features]
y_val_tune = df_tr[val_mask]["y"]

dtrain_tune = xgb.DMatrix(X_train_tune, label=y_train_tune)
dval_tune = xgb.DMatrix(X_val_tune, label=y_val_tune)

# Fixed training settings
NUM_BOOST_ROUND = 2000
EARLY_STOPPING = 100

# Objective for optuna
def mae_from_pred(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def objective(trial):
    param = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "seed": RANDOM_SEED,
        "verbosity": 0,
        "learning_rate": 0.03,
        "max_depth": trial.suggest_int("max_depth", 6, 12),
        "min_child_weight": trial.suggest_int("min_child_weight", 10, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "lambda": trial.suggest_float("lambda", 0.1, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-2, 10.0, log=True),
    }
    evals_result = {}
    bst = xgb.train(
        param,
        dtrain_tune,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dtrain_tune, "train"), (dval_tune, "val")],
        early_stopping_rounds=EARLY_STOPPING,
        evals_result=evals_result,
        verbose_eval=False,
    )
    best_iter = getattr(bst, "best_iteration", None)
    try:
        if best_iter is not None:
            preds = bst.predict(dval_tune, iteration_range=(0, int(best_iter) + 1))
        else:
            preds = bst.predict(dval_tune)
    except TypeError:
        if best_iter is not None:
            preds = bst.predict(dval_tune, ntree_limit=int(best_iter) + 1)
        else:
            preds = bst.predict(dval_tune)
    val_mae = mae_from_pred(np.asarray(y_val_tune), preds)
    trial.report(val_mae, 0)
    return val_mae

# Run Optuna study
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
print(f"Starting Optuna study with {TRIALS} trials (this may take several minutes).")
t0 = time.time()
study.optimize(objective, n_trials=TRIALS, show_progress_bar=True)
t1 = time.time()
print(f"Optuna tuning done in {(t1-t0)/60:.2f} minutes")

best = study.best_trial
print("\nBest trial:")
print("  Value (Val MAE):", best.value)
print("  Params:", best.params)

best_params = best.params.copy()
best_params.update({
    "objective": "reg:squarederror",
    "learning_rate": 0.03,
    "tree_method": "hist",
    "verbosity": 1,
    "seed": RANDOM_SEED
})
best_path = MODEL_DIR / "xgb_optuna_best_params_stage3.json"
with open(best_path, "w") as f:
    json.dump(best_params, f, indent=2)
print("Saved best params to:", best_path)

# Retrain best on Fold2 and evaluate across folds/holdouts
dtrain_best = xgb.DMatrix(X_train_tune, label=y_train_tune)
dval_best = xgb.DMatrix(X_val_tune, label=y_val_tune)

bst_best = xgb.train(
    best_params,
    dtrain_best,
    num_boost_round=NUM_BOOST_ROUND,
    evals=[(dtrain_best, "train"), (dval_best, "val")],
    early_stopping_rounds=EARLY_STOPPING,
    verbose_eval=100
)
best_iter = getattr(bst_best, "best_iteration", None)
print("Best iteration on retrain:", best_iter)

def safe_predict(booster, X_df, best_it):
    dmat = xgb.DMatrix(X_df[features])
    try:
        if best_it is not None:
            return booster.predict(dmat, iteration_range=(0, int(best_it) + 1))
        else:
            return booster.predict(dmat)
    except TypeError:
        if best_it is not None:
            return booster.predict(dmat, ntree_limit=int(best_it) + 1)
        else:
            return booster.predict(dmat)

# Fold2 val
pred_val2 = safe_predict(bst_best, X_val_tune, best_iter)
mae_val2 = mean_absolute_error(y_val_tune, pred_val2)
rmse_val2 = math.sqrt(mean_squared_error(y_val_tune, pred_val2))
r2_val2 = r2_score(y_val_tune, pred_val2)
print(f"\nFold2 (2022) — MAE: {mae_val2:.2f}, RMSE: {rmse_val2:.2f}, R2: {r2_val2:.4f}")

# Fold1 val
X_val1 = df_tr[df_tr["Year"] == 2021][features]
y_val1 = df_tr[df_tr["Year"] == 2021]["y"]
pred_val1 = safe_predict(bst_best, X_val1, best_iter)
mae_val1 = mean_absolute_error(y_val1, pred_val1)
rmse_val1 = math.sqrt(mean_squared_error(y_val1, pred_val1))
r2_val1 = r2_score(y_val1, pred_val1)
print(f"Fold1 (2021) — MAE: {mae_val1:.2f}, RMSE: {rmse_val1:.2f}, R2: {r2_val1:.4f}")

# Val2023 and Test2024
try:
    X_val2023 = X_val_imputed.copy(); y_val2023_arr = np.asarray(y_val)
    X_test2024 = X_test_imputed.copy(); y_test2024_arr = np.asarray(y_test)
    pred_val2023 = safe_predict(bst_best, X_val2023, best_iter)
    pred_test2024 = safe_predict(bst_best, X_test2024, best_iter)
    mae_2023 = mean_absolute_error(y_val2023_arr, pred_val2023)
    r2_2023 = r2_score(y_val2023_arr, pred_val2023)
    mae_2024 = mean_absolute_error(y_test2024_arr, pred_test2024)
    r2_2024 = r2_score(y_test2024_arr, pred_test2024)
    print(f"\nVal 2023 — MAE: {mae_2023:.2f}, R2: {r2_2023:.4f}")
    print(f"Test 2024 — MAE: {mae_2024:.2f}, R2: {r2_2024:.4f}")
except Exception as e:
    print("Could not evaluate on Val2023/Test2024:", e)

# Save best booster & wrapper
booster_path = MODEL_DIR / f"xgb_booster_stage3_optuna_best.json"
bst_best.save_model(str(booster_path))
wrapper = {"booster_path": str(booster_path), "features": features, "best_params": best_params, "best_iter": best_iter}
joblib_path = MODEL_DIR / f"xgb_wrapper_stage3_optuna_best.joblib"
import joblib
joblib.dump(wrapper, joblib_path)
print("\nSaved booster and wrapper to model dir:", MODEL_DIR)

print("\nTuning complete.")


Rows in X_train_imputed (nX): 137755
Using y from: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\y_train_stage2.csv  (length = 137755 )


[I 2025-11-19 07:55:06,566] A new study created in memory with name: no-name-8e9cee87-ae41-4e03-9cae-cf6fca1bd722


Starting Optuna study with 40 trials (this may take several minutes).


Best trial: 0. Best value: 14827.2:   2%|▎         | 1/40 [00:02<01:48,  2.77s/it]

[I 2025-11-19 07:55:09,342] Trial 0 finished with value: 14827.151588676428 and parameters: {'max_depth': 8, 'min_child_weight': 191, 'subsample': 0.892797576724562, 'colsample_bytree': 0.8394633936788146, 'lambda': 0.20513382630874505, 'alpha': 0.029375384576328288}. Best is trial 0 with value: 14827.151588676428.


...

Best trial: 35. Best value: 14665.3:  98%|█████████▊| 39/40 [02:10<00:03,  3.55s/it]

[I 2025-11-19 07:57:17,280] Trial 38 finished with value: 14798.926435818565 and parameters: {'max_depth': 7, 'min_child_weight': 21, 'subsample': 0.8991470739047311, 'colsample_bytree': 0.7524323076423147, 'lambda': 0.13943841394158812, 'alpha': 3.6743625694782147}. Best is trial 35 with value: 14665.33967085149.


Best trial: 35. Best value: 14665.3: 100%|██████████| 40/40 [02:14<00:00,  3.35s/it]

[I 2025-11-19 07:57:20,699] Trial 39 finished with value: 14729.204848074947 and parameters: {'max_depth': 8, 'min_child_weight': 40, 'subsample': 0.9672341261576703, 'colsample_bytree': 0.7227385799967678, 'lambda': 0.21602296014335068, 'alpha': 6.964786116772577}. Best is trial 35 with value: 14665.33967085149.
Optuna tuning done in 2.24 minutes

Best trial:
  Value (Val MAE): 14665.33967085149
  Params: {'max_depth': 8, 'min_child_weight': 29, 'subsample': 0.8846028785366786, 'colsample_bytree': 0.6834869949329374, 'lambda': 0.10226356823769901, 'alpha': 4.830154179135389}
Saved best params to: kcet_ml_project\models\xgboost_stage3\xgb_optuna_best_params_stage3.json
[0]	train-rmse:45015.87263	val-rmse:42951.54567


[100]	train-rmse:20285.62967	val-rmse:21366.47110
[200]	train-rmse:19100.14314	val-rmse:21603.00814
[210]	train-rmse:19029.04028	val-rmse:21624.69678
Best iteration on retrain: 110

Fold2 (2022) — MAE: 14665.34, RMSE: 21357.94, R2: 0.7621
Fold1 (2021) — MAE: 13920.61, RMSE: 20006.54, R2: 0.8253
Could not evaluate on Val2023/Test2024: Found input variables with inconsistent numbers of samples: [52586, 60681]

Saved booster and wrapper to model dir: kcet_ml_project\models\xgboost_stage3

Tuning complete.


In [14]:
# Cell 8 (fixed): Final retrain on 2020-2023 by concatenating Train(2020-22) + Val(2023), evaluate on Test(2024)
import os, json, time, math
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Paths
PROJECT_ROOT = Path(".")
MODEL_DIR = PROJECT_ROOT / "kcet_ml_project" / "models" / "xgboost_stage3"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Load best params
best_params_path = MODEL_DIR / "xgb_optuna_best_params_stage3.json"
if best_params_path.exists():
    with open(best_params_path) as f:
        best_params = json.load(f)
    print("Loaded best params from:", best_params_path)
else:
    raise RuntimeError("Best params not found. Run tuning first.")

print("Best params summary:", {k: best_params[k] for k in ["max_depth","min_child_weight","subsample","colsample_bytree","lambda","alpha"] if k in best_params})

# Ensure required dfs and targets exist
try:
    Xtr = X_train_imputed.copy()
    Xv = X_val_imputed.copy()
    Xt = X_test_imputed.copy()
    ytr = y_train.copy()
    yv = y_val.copy()
    yt = y_test.copy()
except NameError as e:
    raise RuntimeError("Required X/y objects not found in memory. Run previous cells to load imputed X and y.") from e

print("Years present in X_train_imputed:", sorted(Xtr["Year"].unique()))
print("Years present in X_val_imputed:  ", sorted(Xv["Year"].unique()))
print("Years present in X_test_imputed: ", sorted(Xt["Year"].unique()))

# Check shapes align for concatenation
if len(Xtr) != len(ytr):
    print(f"Warning: X_train_imputed rows {len(Xtr)} != y_train length {len(ytr)}. Trying to align by index...")

# Concatenate train+val to create final training set (2020-2023)
X_final_train = pd.concat([Xtr, Xv], axis=0).reset_index(drop=True)
y_final_train = pd.concat([pd.Series(ytr).reset_index(drop=True), pd.Series(yv).reset_index(drop=True)], axis=0).reset_index(drop=True)

print("Concatenated final train shape:", X_final_train.shape, "-> y length:", len(y_final_train))
# Verify Year coverage
print("Final train Years:", sorted(X_final_train["Year"].unique()))

# Build early-stop split: use the rows with Year==2023 as the validation for early stopping
mask_val2023 = X_final_train["Year"] == 2023
if mask_val2023.sum() == 0:
    raise RuntimeError("After concatenation there are still 0 rows for Year==2023. Aborting.")
X_fit = X_final_train[~mask_val2023].reset_index(drop=True)   # 2020-2022
y_fit = y_final_train[~mask_val2023].values
X_earlyval = X_final_train[mask_val2023].reset_index(drop=True)  # 2023
y_earlyval = y_final_train[mask_val2023].values

print(" - training rows (2020-2022):", len(X_fit))
print(" - early-stop validation rows (2023):", len(X_earlyval))

# DMatrix
features = locked_features
dtrain = xgb.DMatrix(X_fit[features], label=y_fit)
dval = xgb.DMatrix(X_earlyval[features], label=y_earlyval)
dtest = xgb.DMatrix(Xt[features], label=np.asarray(yt))

# Train final booster
num_boost_round = 5000
early_stopping_rounds = 300

print("\nStarting final training (train 2020-2022, early-stop on 2023)...")
t0 = time.time()
bst_final = xgb.train(
    best_params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=early_stopping_rounds,
    verbose_eval=100
)
t1 = time.time()
print(f"Training completed in {(t1-t0):.1f}s")
best_iter = getattr(bst_final, "best_iteration", None)
print("Best iteration:", best_iter)

# Predictions on test
def safe_predict(booster, dmatrix, best_it):
    try:
        if best_it is not None:
            return booster.predict(dmatrix, iteration_range=(0, int(best_it)+1))
        else:
            return booster.predict(dmatrix)
    except TypeError:
        if best_it is not None:
            return booster.predict(dmatrix, ntree_limit=int(best_it)+1)
        else:
            return booster.predict(dmatrix)

y_test_pred = safe_predict(bst_final, dtest, best_iter)
y_test_true = np.asarray(yt)

test_mae = mean_absolute_error(y_test_true, y_test_pred)
test_rmse = math.sqrt(mean_squared_error(y_test_true, y_test_pred))
test_r2 = r2_score(y_test_true, y_test_pred)

print("\nFINAL TEST RESULTS (2024):")
print(f"  MAE:  {test_mae:.2f}")
print(f"  RMSE: {test_rmse:.2f}")
print(f"  R²:   {test_r2:.4f}")

# Save artifacts
booster_path = MODEL_DIR / f"xgb_booster_stage3_final_bestiter{best_iter if best_iter is not None else 'none'}.json"
bst_final.save_model(str(booster_path))
wrapper = {
    "booster_path": str(booster_path),
    "features": features,
    "best_params": best_params,
    "best_iter": best_iter
}
joblib_path = MODEL_DIR / "xgb_wrapper_stage3_final.joblib"
joblib.dump(wrapper, joblib_path)
np.save(MODEL_DIR / "y_test_pred_xgb_stage3_final.npy", y_test_pred)
meta = {
    "best_params": best_params,
    "best_iter": best_iter,
    "test_mae": float(test_mae),
    "test_rmse": float(test_rmse),
    "test_r2": float(test_r2)
}
with open(MODEL_DIR / "xgb_final_meta_stage3.json", "w") as f:
    json.dump(meta, f, indent=2)

print("\nSaved final booster, wrapper, predictions, and meta to:", MODEL_DIR)
print("Cell complete.")


Loaded best params from: kcet_ml_project\models\xgboost_stage3\xgb_optuna_best_params_stage3.json
Best params summary: {'max_depth': 8, 'min_child_weight': 29, 'subsample': 0.8846028785366786, 'colsample_bytree': 0.6834869949329374, 'lambda': 0.10226356823769901, 'alpha': 4.830154179135389}
Years present in X_train_imputed: [2020, 2021, 2022]
Years present in X_val_imputed:   [2023]
Years present in X_test_imputed:  [2024]
Concatenated final train shape: (198436, 32) -> y length: 137755
Final train Years: [2020, 2021, 2022, 2023]
 - training rows (2020-2022): 137755
 - early-stop validation rows (2023): 60681

Starting final training (train 2020-2022, early-stop on 2023)...


XGBoostError: [08:19:54] C:\actions-runner\_work\xgboost\xgboost\src\metric\elementwise_metric.cu:361: Check failed: preds.Size() == info.labels.Size() (60681 vs. 0) : label and prediction size not match, hint: use merror or mlogloss for multi-class classification

In [18]:
# Fix-and-run final retrain: reload y_train from disk (robust), then two-step final training
import os, json, time, math
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Paths
PROJECT_ROOT = Path(".")
MODEL_DIR = PROJECT_ROOT / "kcet_ml_project" / "models" / "xgboost_stage3"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
candidate_dirs = [
    Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected"),
    PROJECT_ROOT / "kcet_ml_project" / "data",
    Path("/mnt/data"),
]

# find y_train_stage2.csv (or similar) and load it
y_paths = []
for d in candidate_dirs:
    if not d.exists(): 
        continue
    y_paths.extend(list(d.glob("y_train*.csv")))
    y_paths.extend(list(d.glob("*y_train*.csv")))
    y_paths.extend(list(d.glob("y_train_stage2.csv")))

y_paths = sorted(set(y_paths))
print("Candidate y files found:", [str(p) for p in y_paths])

# require X_train_imputed present
try:
    Xtr = X_train_imputed.copy()
except NameError:
    raise RuntimeError("X_train_imputed not found in memory. Run earlier cells to prepare imputed Xs.")

nX = len(Xtr)
print("X_train_imputed rows:", nX)

found = False
for p in y_paths:
    try:
        tmp = pd.read_csv(p)
        if tmp.shape[1] == 1:
            arr = tmp.iloc[:,0].values
        else:
            # try to infer label column
            candidates = [c for c in tmp.columns if c.lower() in ("y","target","cutoff","label","score")]
            if candidates:
                arr = tmp[candidates[0]].values
            else:
                numcols = tmp.select_dtypes(include=[np.number]).columns.tolist()
                if numcols:
                    arr = tmp[numcols[0]].values
                else:
                    continue
        print(f"Loaded {p} with length {len(arr)}")
        if len(arr) == nX:
            y_train_arr = np.asarray(arr)
            print("Selected as y_train:", p)
            found = True
            break
        else:
            print("  -> length mismatch, skipping this file.")
    except Exception as e:
        print("  -> could not read", p, e)

if not found:
    # check in-memory y_train/ytr_all etc.
    for cand in ("y_train","ytr_all","y_all"):
        if cand in globals():
            arr = np.asarray(globals()[cand])
            print(f"In-memory candidate {cand} length {len(arr)}")
            if len(arr) == nX:
                y_train_arr = arr
                print("Selected in-memory", cand, "as y_train")
                found = True
                break

if not found:
    raise RuntimeError("Could not locate a y_train vector matching X_train_imputed length. Place the correct y_train CSV in one of the candidate dirs or ensure y_train in memory matches X_train_imputed rows.")

# Now ensure y_val and y_test exist and are aligned; prefer in-memory variables
if 'y_val' in globals():
    y_val_arr = np.asarray(y_val)
else:
    # try to find val file
    y_val_arr = None

if 'y_test' in globals():
    y_test_arr = np.asarray(y_test)
else:
    y_test_arr = None

print("Loaded lengths -> y_train:", len(y_train_arr), 
      "y_val:", (len(y_val_arr) if y_val_arr is not None else None),
      "y_test:", (len(y_test_arr) if y_test_arr is not None else None))

# Load best params
best_params_path = MODEL_DIR / "xgb_optuna_best_params_stage3.json"
if not best_params_path.exists():
    raise RuntimeError("Best params not found. Run Optuna tuning first.")
with open(best_params_path) as f:
    best_params = json.load(f)
print("Best params loaded.")

# Prepare DMatrices for Step1:
# Xtr -> should contain rows for years 2020-2022 (your Stage2 training set)
# Xv  -> contains 2023 (your Stage2 val)
try:
    Xv = X_val_imputed.copy()
    Xt = X_test_imputed.copy()
except NameError:
    raise RuntimeError("X_val_imputed or X_test_imputed missing from memory. Run the earlier cells.")

# Step1 DMatrix (train on Xtr, early-stop on Xv)
dtrain_step1 = xgb.DMatrix(Xtr[locked_features], label=y_train_arr)
if len(Xv) != 0 and ('y_val' in globals() or 'y_val_arr' in locals()):
    # ensure y_val_arr aligns
    if 'y_val' in globals():
        if len(y_val) != len(Xv):
            raise RuntimeError(f"y_val length mismatch: len(y_val)={len(y_val)} vs len(X_val_imputed)={len(Xv)}. Fix alignment before proceeding.")
        y_val_arr = np.asarray(y_val)
    dval_step1 = xgb.DMatrix(Xv[locked_features], label=y_val_arr)
else:
    raise RuntimeError("X_val_imputed or y_val not available/empty; cannot perform early stopping step.")

# Step 1: find best_iter
num_boost_round = 5000
early_stopping_rounds = 300
print("\nStep1: training on X_train_imputed (2020-22) with early-stop on X_val_imputed (2023) ...")
bst_step1 = xgb.train(
    best_params,
    dtrain_step1,
    num_boost_round=num_boost_round,
    evals=[(dtrain_step1, "train"), (dval_step1, "val")],
    early_stopping_rounds=early_stopping_rounds,
    verbose_eval=100
)
best_iter = getattr(bst_step1, "best_iteration", None)
print("Found best_iteration:", best_iter)

# Evaluate Step1 val metrics
def safe_pred(booster, X_df, best_it):
    dmat = xgb.DMatrix(X_df[locked_features])
    try:
        if best_it is not None:
            return booster.predict(dmat, iteration_range=(0, int(best_it)+1))
        else:
            return booster.predict(dmat)
    except TypeError:
        if best_it is not None:
            return booster.predict(dmat, ntree_limit=int(best_it)+1)
        else:
            return booster.predict(dmat)

pred_val_step1 = safe_pred(bst_step1, Xv, best_iter)
mae_val_step1 = mean_absolute_error(np.asarray(y_val), pred_val_step1)
print(f"Step1 Val MAE: {mae_val_step1:.2f}")

# Step2: Retrain on full 2020-2023 for best_iter+1 rounds
print("\nStep2: retrain on concatenated X_train_imputed + X_val_imputed for best_iter rounds (no early stopping)")
X_full = pd.concat([Xtr.reset_index(drop=True), Xv.reset_index(drop=True)], axis=0).reset_index(drop=True)
y_full = np.concatenate([y_train_arr, np.asarray(y_val)], axis=0)
if len(X_full) != len(y_full):
    raise RuntimeError("Length mismatch after concatenation. Aborting.")

dtrain_final = xgb.DMatrix(X_full[locked_features], label=y_full)
dtest = xgb.DMatrix(Xt[locked_features], label=np.asarray(y_test) if 'y_test' in globals() else None)

if best_iter is None:
    best_iter = 200
print("Retraining final model for", int(best_iter)+1, "rounds.")
bst_final = xgb.train(best_params, dtrain_final, num_boost_round=int(best_iter)+1, verbose_eval=100)

# Predict on test
y_test_pred = safe_pred(bst_final, Xt, best_iter)
if 'y_test' in globals():
    test_mae = mean_absolute_error(np.asarray(y_test), y_test_pred)
    test_rmse = math.sqrt(mean_squared_error(np.asarray(y_test), y_test_pred))
    test_r2 = r2_score(np.asarray(y_test), y_test_pred)
    print("\nFinal Test metrics:")
    print(" MAE:", round(test_mae,2), "RMSE:", round(test_rmse,2), "R2:", round(test_r2,4))
else:
    print("Saved predictions but no y_test in memory to compute metrics.")

# Save artifacts
booster_path = MODEL_DIR / f"xgb_booster_stage3_final_bestiter{best_iter}.json"
bst_final.save_model(str(booster_path))
joblib.dump({"booster_path":str(booster_path),"features":locked_features,"best_params":best_params,"best_iter":int(best_iter)}, MODEL_DIR / "xgb_wrapper_stage3_final.joblib")
np.save(MODEL_DIR / "y_test_pred_xgb_stage3_final.npy", y_test_pred)
with open(MODEL_DIR / "xgb_final_meta_stage3.json","w") as f:
    json.dump({
        "best_iter": int(best_iter),
        "test_mae": float(test_mae) if 'test_mae' in locals() else None,
        "test_rmse": float(test_rmse) if 'test_rmse' in locals() else None,
        "test_r2": float(test_r2) if 'test_r2' in locals() else None
    }, f, indent=2)

print("\nSaved final artifacts to:", MODEL_DIR)


Candidate y files found: ['D:\\Courses\\Global Academy of Technology\\kcet-college-pred\\notebooks\\kcet_ml_project\\data\\stage2_v2_corrected\\y_train_stage2.csv', 'kcet_ml_project\\data\\y_train_stage2.csv']
X_train_imputed rows: 137755
Loaded D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected\y_train_stage2.csv with length 137755
Selected as y_train: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected\y_train_stage2.csv
Loaded lengths -> y_train: 137755 y_val: 52586 y_test: 71626
Best params loaded.


RuntimeError: y_val length mismatch: len(y_val)=52586 vs len(X_val_imputed)=60681. Fix alignment before proceeding.

In [19]:
# FIX: auto-find matching y_val / y_test csvs, align them, then two-step final retrain & eval
import os, json, time, math
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------- Config ----------
PROJECT_ROOT = Path(".")
MODEL_DIR = PROJECT_ROOT / "kcet_ml_project" / "models" / "xgboost_stage3"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

candidate_dirs = [
    Path(r"D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected"),
    PROJECT_ROOT / "kcet_ml_project" / "data",
    Path("/mnt/data"),
    Path(".")
]

# helper to find a y file matching a target length
def find_matching_y(target_length, patterns=["y_val*.csv","*y_val*.csv","y_test*.csv","*y_test*.csv","val_*y*.csv","test_*y*.csv","y_*.csv"]):
    candidates = []
    for d in candidate_dirs:
        if not d.exists():
            continue
        for pat in patterns:
            for p in d.glob(pat):
                if p in candidates:
                    continue
                candidates.append(p)
    # load & check lengths
    for p in sorted(set(candidates)):
        try:
            tmp = pd.read_csv(p)
            if tmp.shape[1] == 1:
                arr = tmp.iloc[:,0].values
            else:
                # prefer obvious columns
                col = None
                for c in tmp.columns:
                    if c.lower() in ("y","target","cutoff","label","score"):
                        col = c; break
                if col is not None:
                    arr = tmp[col].values
                else:
                    numeric_cols = tmp.select_dtypes(include=[np.number]).columns.tolist()
                    if numeric_cols:
                        arr = tmp[numeric_cols[0]].values
                    else:
                        continue
            if len(arr) == target_length:
                return np.asarray(arr), p
        except Exception:
            continue
    return None, None

# Ensure X_* imputed exist
try:
    Xtr = X_train_imputed.copy()
    Xv  = X_val_imputed.copy()
    Xt  = X_test_imputed.copy()
except NameError:
    raise RuntimeError("Imputed X dataframes (X_train_imputed, X_val_imputed, X_test_imputed) missing from memory. Run earlier cells.")

print("Shapes: Xtr, Xv, Xt =", Xtr.shape, Xv.shape, Xt.shape)

# find matching y_val
target_len_val = len(Xv)
y_val_arr, y_val_path = find_matching_y(target_len_val, patterns=["y_val*.csv","*y_val*.csv","val_*y*.csv","*val*.csv","y_*.csv"])
if y_val_arr is None:
    # try exact filename
    for d in candidate_dirs:
        p = d / "y_val_stage2.csv"
        if p.exists():
            try:
                tmp = pd.read_csv(p)
                if tmp.shape[1]==1:
                    arr = tmp.iloc[:,0].values
                else:
                    col_candidates=[c for c in tmp.columns if c.lower() in ("y","target","cutoff","label","score")]
                    if col_candidates:
                        arr = tmp[col_candidates[0]].values
                    else:
                        arr = tmp.select_dtypes(include=[np.number]).iloc[:,0].values
                if len(arr)==target_len_val:
                    y_val_arr = np.asarray(arr); y_val_path = p
            except Exception:
                pass

if y_val_arr is None:
    raise RuntimeError(f"Could not find a y_val CSV matching X_val_imputed length ({target_len_val}). Searched: {candidate_dirs}. Please supply the correct y_val file.")

print("Using y_val from:", y_val_path, "length:", len(y_val_arr))

# find matching y_test
target_len_test = len(Xt)
y_test_arr, y_test_path = find_matching_y(target_len_test, patterns=["y_test*.csv","*y_test*.csv","test_*y*.csv","*test*.csv","y_*.csv"])
if y_test_arr is None:
    # try exact filename
    for d in candidate_dirs:
        p = d / "y_test_stage2.csv"
        if p.exists():
            try:
                tmp = pd.read_csv(p)
                if tmp.shape[1]==1:
                    arr = tmp.iloc[:,0].values
                else:
                    col_candidates=[c for c in tmp.columns if c.lower() in ("y","target","cutoff","label","score")]
                    if col_candidates:
                        arr = tmp[col_candidates[0]].values
                    else:
                        arr = tmp.select_dtypes(include=[np.number]).iloc[:,0].values
                if len(arr)==target_len_test:
                    y_test_arr = np.asarray(arr); y_test_path = p
            except Exception:
                pass

if y_test_arr is None:
    raise RuntimeError(f"Could not find a y_test CSV matching X_test_imputed length ({target_len_test}). Searched: {candidate_dirs}. Please supply the correct y_test file.")

print("Using y_test from:", y_test_path, "length:", len(y_test_arr))

# load y_train (ensure it's aligned)
target_len_train = len(Xtr)
y_train_arr = None
# try y_train_stage2.csv first
for d in candidate_dirs:
    p = d / "y_train_stage2.csv"
    if p.exists():
        try:
            tmp = pd.read_csv(p)
            if tmp.shape[1]==1:
                arr = tmp.iloc[:,0].values
            else:
                col_candidates=[c for c in tmp.columns if c.lower() in ("y","target","cutoff","label","score")]
                if col_candidates:
                    arr = tmp[col_candidates[0]].values
                else:
                    arr = tmp.select_dtypes(include=[np.number]).iloc[:,0].values
            if len(arr)==target_len_train:
                y_train_arr = np.asarray(arr)
                y_train_path = p
                break
        except Exception:
            pass
if y_train_arr is None:
    # try other y_train patterns
    y_train_arr, y_train_path = find_matching_y(target_len_train, patterns=["y_train*.csv","*y_train*.csv","y_*.csv"])
if y_train_arr is None:
    # fallback to in-memory if matches length
    if 'y_train' in globals() and len(np.asarray(y_train))==target_len_train:
        y_train_arr = np.asarray(y_train); y_train_path = "in-memory:y_train"
    else:
        raise RuntimeError(f"Could not locate y_train of length {target_len_train}. Please ensure y_train CSV exists in candidate directories.")

print("Using y_train from:", y_train_path, "length:", len(y_train_arr))

# Save these aligned arrays to memory variables (overwrite dirty ones)
y_train = y_train_arr
y_val = y_val_arr
y_test = y_test_arr

# Load best params file
best_params_path = MODEL_DIR / "xgb_optuna_best_params_stage3.json"
if not best_params_path.exists():
    raise RuntimeError("Best params file missing. Run tuning first.")
with open(best_params_path) as f:
    best_params = json.load(f)
print("Loaded best params.")

# Step 1: find best_iter (train on Xtr, early-stop on Xv)
features = locked_features
dtrain_step1 = xgb.DMatrix(Xtr[features], label=y_train)
dval_step1 = xgb.DMatrix(Xv[features], label=y_val)

num_boost_round = 5000
early_stopping_rounds = 300

print("\nStep1: training (2020-2022) with early-stop on 2023 ...")
bst_step1 = xgb.train(
    best_params,
    dtrain_step1,
    num_boost_round=num_boost_round,
    evals=[(dtrain_step1, "train"), (dval_step1, "val")],
    early_stopping_rounds=early_stopping_rounds,
    verbose_eval=100
)
best_iter = getattr(bst_step1, "best_iteration", None)
print("Found best_iteration:", best_iter)

# Step1 val metrics
pred_val = bst_step1.predict(dval_step1) if best_iter is None else bst_step1.predict(dval_step1, iteration_range=(0, int(best_iter)+1))
mae_val1 = mean_absolute_error(y_val, pred_val)
print("Step1 Val MAE:", round(mae_val1,2))

# Step 2: retrain on full 2020-2023 for best_iter rounds
print("\nStep2: retrain on full train (concat Xtr+Xv) for best_iter rounds")
X_full = pd.concat([Xtr.reset_index(drop=True), Xv.reset_index(drop=True)], axis=0).reset_index(drop=True)
y_full = np.concatenate([y_train, y_val], axis=0)
if len(X_full) != len(y_full):
    raise RuntimeError("Length mismatch after concat X_full vs y_full.")

dtrain_final = xgb.DMatrix(X_full[features], label=y_full)
dtest = xgb.DMatrix(Xt[features], label=y_test)

if best_iter is None:
    best_iter = 200
print("Retraining for", int(best_iter)+1, "rounds.")
bst_final = xgb.train(best_params, dtrain_final, num_boost_round=int(best_iter)+1, verbose_eval=100)

# Predict on test
pred_test = bst_final.predict(dtest) if best_iter is None else bst_final.predict(dtest, iteration_range=(0, int(best_iter)+1))
test_mae = mean_absolute_error(y_test, pred_test)
test_rmse = math.sqrt(mean_squared_error(y_test, pred_test))
test_r2 = r2_score(y_test, pred_test)
print("\nFINAL TEST RESULTS (2024):")
print(" MAE:", round(test_mae,2), "RMSE:", round(test_rmse,2), "R2:", round(test_r2,4))

# Save artifacts
booster_path = MODEL_DIR / f"xgb_booster_stage3_final_bestiter{best_iter}.json"
bst_final.save_model(str(booster_path))
joblib.dump({"booster_path":str(booster_path),"features":features,"best_params":best_params,"best_iter":int(best_iter)}, MODEL_DIR / "xgb_wrapper_stage3_final.joblib")
np.save(MODEL_DIR / "y_test_pred_xgb_stage3_final.npy", pred_test)
with open(MODEL_DIR / "xgb_final_meta_stage3.json","w") as f:
    json.dump({"best_iter":int(best_iter),"test_mae":float(test_mae),"test_rmse":float(test_rmse),"test_r2":float(test_r2)}, f, indent=2)

print("\nSaved final artifacts to:", MODEL_DIR)


Shapes: Xtr, Xv, Xt = (137755, 32) (60681, 32) (71626, 32)
Using y_val from: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected\val_stage2_final.csv length: 60681
Using y_test from: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected\test_stage2_final.csv length: 71626
Using y_train from: D:\Courses\Global Academy of Technology\kcet-college-pred\notebooks\kcet_ml_project\data\stage2_v2_corrected\y_train_stage2.csv length: 137755
Loaded best params.

Step1: training (2020-2022) with early-stop on 2023 ...
[0]	train-rmse:44224.91511	val-rmse:52104.08823
[100]	train-rmse:19492.33629	val-rmse:28685.19532
[200]	train-rmse:18284.46478	val-rmse:27619.51165
[300]	train-rmse:17729.94370	val-rmse:27467.36378
[400]	train-rmse:17408.38868	val-rmse:27363.58285
[500]	train-rmse:17089.83560	val-rmse:27303.68953
[600]	train-rmse:16824.72388	val-rmse:27268.93755
[700]	train-rmse:16587.3422

In [20]:
# Cell: SHAP explainability for final XGBoost model (Val & Test)
import os, json, math, time
from pathlib import Path
import numpy as np
import pandas as pd

# try import shap
try:
    import shap
except Exception:
    print("shap not found — installing shap (may take 30-60s)...")
    import sys
    !{sys.executable} -m pip install -q shap
    import shap

import xgboost as xgb
import joblib
from sklearn.utils import check_random_state

# Paths
PROJECT_ROOT = Path(".")
MODEL_DIR = PROJECT_ROOT / "kcet_ml_project" / "models" / "xgboost_stage3"
OUT_DIR = MODEL_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load wrapper to get feature list and booster path
wrapper_candidates = list(MODEL_DIR.glob("xgb_wrapper_stage3_final*.joblib")) + list(MODEL_DIR.glob("xgb_wrapper_stage3_optuna_best.joblib")) + list(MODEL_DIR.glob("xgb_wrapper_stage3_best*.joblib"))
if not wrapper_candidates:
    raise RuntimeError("Could not find wrapper joblib in model dir.")
wrapper = joblib.load(wrapper_candidates[0])
features = wrapper["features"]
booster_path = wrapper["booster_path"]
print("Using wrapper:", wrapper_candidates[0].name)
print("Booster path:", booster_path)
print("Number of features:", len(features))

# load booster
bst = xgb.Booster()
bst.load_model(str(booster_path))

# Load X_val and X_test (imputed) from memory
try:
    X_val = X_val_imputed.copy()
    X_test = X_test_imputed.copy()
except NameError:
    raise RuntimeError("X_val_imputed / X_test_imputed not in memory. Run earlier cells that created imputed datasets.")

# Sample rows (cap to 20k each) for SHAP to control memory/time
rng = check_random_state(42)
def sample_df(df, n=20000):
    if len(df) <= n:
        return df
    return df.sample(n=n, random_state=rng)

Xv_sample = sample_df(X_val[features], n=20000)
Xt_sample = sample_df(X_test[features], n=20000)

print("SHAP samples: Val", Xv_sample.shape, "Test", Xt_sample.shape)

# build TreeExplainer
explainer = shap.TreeExplainer(bst, feature_perturbation="tree_path_dependent")  # safe for XGBoost
t0 = time.time()
shap_v = explainer.shap_values(Xv_sample)  # shape (n, n_features)
shap_t = explainer.shap_values(Xt_sample)
t1 = time.time()
print(f"Computed SHAP values in {(t1-t0):.1f}s")

# Save raw SHAP arrays (numpy)
np.save(OUT_DIR / "shap_val.npy", shap_v)
np.save(OUT_DIR / "shap_test.npy", shap_t)
print("Saved SHAP arrays to:", OUT_DIR)

# Compute mean absolute SHAP per feature and rank
def mean_abs_shap_table(shap_arr, df_sample):
    mean_abs = np.mean(np.abs(shap_arr), axis=0)
    feat = list(df_sample.columns)
    df_shap = pd.DataFrame({"feature": feat, "mean_abs_shap": mean_abs})
    df_shap = df_shap.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    return df_shap

df_shap_val = mean_abs_shap_table(shap_v, Xv_sample)
df_shap_test = mean_abs_shap_table(shap_t, Xt_sample)

# Save CSVs
df_shap_val.to_csv(OUT_DIR / "shap_mean_abs_val.csv", index=False)
df_shap_test.to_csv(OUT_DIR / "shap_mean_abs_test.csv", index=False)

# Print top-15 features for Val and Test
print("\nTop 15 features by mean(|SHAP|) — Validation (2023):")
print(df_shap_val.head(15).to_string(index=False))

print("\nTop 15 features by mean(|SHAP|) — Test (2024):")
print(df_shap_test.head(15).to_string(index=False))

# Quick sanity checks
top_val_feat = df_shap_val.loc[0, "feature"]
top_test_feat = df_shap_test.loc[0, "feature"]
print(f"\nTop feature Val: {top_val_feat}; Top feature Test: {top_test_feat}")

# Is 'Year' dominating?
def is_year_dominant(df_shap, threshold_ratio=1.5):
    top = df_shap.iloc[0]["mean_abs_shap"]
    second = df_shap.iloc[1]["mean_abs_shap"]
    return (df_shap.iloc[0]["feature"].lower() == "year") and (top / max(second, 1e-9) > threshold_ratio)

if is_year_dominant(df_shap_val) or is_year_dominant(df_shap_test):
    print("WARNING: 'Year' is dominating SHAP importance — investigate possible leakage/time-encoding issues.")
else:
    print("OK: 'Year' is not dominating SHAP importance.")

# Optionally, create simple bar plots (saved as PNG)
try:
    import matplotlib.pyplot as plt
    topk = 15
    plt.figure(figsize=(8,6))
    plt.barh(df_shap_val["feature"].head(topk)[::-1], df_shap_val["mean_abs_shap"].head(topk)[::-1])
    plt.title("Top-15 mean|SHAP| — Validation (2023)")
    plt.xlabel("mean(|SHAP|)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "shap_top15_val.png", dpi=150)
    plt.close()

    plt.figure(figsize=(8,6))
    plt.barh(df_shap_test["feature"].head(topk)[::-1], df_shap_test["mean_abs_shap"].head(topk)[::-1])
    plt.title("Top-15 mean|SHAP| — Test (2024)")
    plt.xlabel("mean(|SHAP|)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "shap_top15_test.png", dpi=150)
    plt.close()
    print("Saved SHAP bar plots to PNG.")
except Exception as e:
    print("Could not produce/save plots:", e)

print("\nSHAP run complete. Files written to:", OUT_DIR)


Using wrapper: xgb_wrapper_stage3_final.joblib
Booster path: kcet_ml_project\models\xgboost_stage3\xgb_booster_stage3_final_bestiter713.json
Number of features: 32
SHAP samples: Val (20000, 32) Test (20000, 32)
Computed SHAP values in 231.8s
Saved SHAP arrays to: kcet_ml_project\models\xgboost_stage3

Top 15 features by mean(|SHAP|) — Validation (2023):
                           feature  mean_abs_shap
        Historical_Mean_Percentile   18747.292969
                    Category_Score   13690.161133
           Historical_Mean_Primary    6359.609375
                  cutoff_lag1Y_L1Y    5231.031738
                             Round    4729.557129
             branch_prevY_mean_L1Y    3610.339600
                         Is_Recent    2775.057861
         College_Branch_target_enc    2768.282227
                              Year    2012.701782
                  cutoff_lag2Y_L2Y    1808.036621
                 Branch_Popularity    1445.886108
                Historical_Std_Raw    1392.8

In [21]:
# Step 10 — FULL SEGMENTED ROBUSTNESS DIAGNOSTICS

import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

# ==== Prepare Paths ====
PROJECT_ROOT = Path(".")
MODEL_DIR = PROJECT_ROOT / "kcet_ml_project" / "models" / "xgboost_stage3"
DIAG_DIR = MODEL_DIR / "diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

# ==== Load wrapper and booster ====
wrapper_path = MODEL_DIR / "xgb_wrapper_stage3_final.joblib"
wrapper = joblib.load(wrapper_path)
booster_path = wrapper["booster_path"]
features = wrapper["features"]

bst = xgb.Booster()
bst.load_model(booster_path)

# ==== Make sure X_val, X_test, y_val, y_test exist ====
Xv = X_val_imputed.copy()
Xt = X_test_imputed.copy()
yv = np.asarray(y_val)
yt = np.asarray(y_test)

# ==== SHARED PREDICTION HELPER ====
def predict_df(bst, df):
    dmat = xgb.DMatrix(df[features])
    return bst.predict(dmat)

# VAL & TEST predictions
Xv["y_true"] = yv
Xv["y_pred"] = predict_df(bst, Xv)

Xt["y_true"] = yt
Xt["y_pred"] = predict_df(bst, Xt)

# ==== Helper: compute MAE per group ====
def mae_by(df, col):
    out = (
        df.groupby(col)
          .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
          .reset_index(name="MAE")
          .sort_values("MAE", ascending=False)
    )
    return out

def save_diag(name, df):
    p = DIAG_DIR / f"{name}.csv"
    df.to_csv(p, index=False)
    print(f"Saved: {p}")

print("\n============= SEGMENT-WISE MAE (VALIDATION 2023) =============\n")

# 1. By Branch
val_branch = mae_by(Xv, "College_Branch_target_enc")
save_diag("val_mae_by_branch", val_branch)

# 2. By Exam_Type
val_exam = mae_by(Xv, "Exam_Type")
save_diag("val_mae_by_examtype", val_exam)

# 3. By Round
val_round = mae_by(Xv, "Round")
save_diag("val_mae_by_round", val_round)

# 4. By College Tier
val_tier = mae_by(Xv, "College_Tier_Numeric")
save_diag("val_mae_by_tier", val_tier)

# 5. By Category Score (binned)
Xv["Category_bin"] = pd.qcut(Xv["Category_Score"], q=5, duplicates="drop")
val_cat = mae_by(Xv, "Category_bin")
save_diag("val_mae_by_category_bin", val_cat)

# 6. By Branch_Popularity (quantiles)
Xv["Popularity_bin"] = pd.qcut(Xv["Branch_Popularity"], q=5, duplicates="drop")
val_pop = mae_by(Xv, "Popularity_bin")
save_diag("val_mae_by_popularity", val_pop)

# 7. Historical Mean Primary strength
Xv["HistPrimary_bin"] = pd.qcut(Xv["Historical_Mean_Primary"], q=5, duplicates="drop")
val_histpri = mae_by(Xv, "HistPrimary_bin")
save_diag("val_mae_by_histprimary", val_histpri)

# 8. Lag1Y strength
Xv["Lag1Y_bin"] = pd.qcut(Xv["cutoff_lag1Y_L1Y"], q=5, duplicates="drop")
val_lag1 = mae_by(Xv, "Lag1Y_bin")
save_diag("val_mae_by_lag1Y", val_lag1)

print("\n============= SEGMENT-WISE MAE (TEST 2024) =============\n")

# Repeat for TEST
Xt["Category_bin"] = pd.qcut(Xt["Category_Score"], q=5, duplicates="drop")
Xt["Popularity_bin"] = pd.qcut(Xt["Branch_Popularity"], q=5, duplicates="drop")
Xt["HistPrimary_bin"] = pd.qcut(Xt["Historical_Mean_Primary"], q=5, duplicates="drop")
Xt["Lag1Y_bin"] = pd.qcut(Xt["cutoff_lag1Y_L1Y"], q=5, duplicates="drop")

test_branch = mae_by(Xt, "College_Branch_target_enc");  save_diag("test_mae_by_branch", test_branch)
test_exam   = mae_by(Xt, "Exam_Type");                  save_diag("test_mae_by_examtype", test_exam)
test_round  = mae_by(Xt, "Round");                      save_diag("test_mae_by_round", test_round)
test_tier   = mae_by(Xt, "College_Tier_Numeric");       save_diag("test_mae_by_tier", test_tier)
test_cat    = mae_by(Xt, "Category_bin");               save_diag("test_mae_by_category_bin", test_cat)
test_pop    = mae_by(Xt, "Popularity_bin");             save_diag("test_mae_by_popularity", test_pop)
test_histpri= mae_by(Xt, "HistPrimary_bin");            save_diag("test_mae_by_histprimary", test_histpri)
test_lag1   = mae_by(Xt, "Lag1Y_bin");                  save_diag("test_mae_by_lag1Y", test_lag1)

print("\nAll segmentation diagnostics saved to:", DIAG_DIR)
print("Step 10 complete.")



============= SEGMENT-WISE MAE (VALIDATION 2023) =============



C:\Users\rahul\AppData\Local\Temp\ipykernel_27768\844501198.py:47: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
C:\Users\rahul\AppData\Local\Temp\ipykernel_27768\844501198.py:47: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
C:\Users\rahul\AppData\Local\Temp\ipykernel_27768\844501198.py:47:

Saved: kcet_ml_project\models\xgboost_stage3\diagnostics\val_mae_by_branch.csv
Saved: kcet_ml_project\models\xgboost_stage3\diagnostics\val_mae_by_examtype.csv
Saved: kcet_ml_project\models\xgboost_stage3\diagnostics\val_mae_by_round.csv
Saved: kcet_ml_project\models\xgboost_stage3\diagnostics\val_mae_by_tier.csv
Saved: kcet_ml_project\models\xgboost_stage3\diagnostics\val_mae_by_category_bin.csv
Saved: kcet_ml_project\models\xgboost_stage3\diagnostics\val_mae_by_popularity.csv
Saved: kcet_ml_project\models\xgboost_stage3\diagnostics\val_mae_by_histprimary.csv
Saved: kcet_ml_project\models\xgboost_stage3\diagnostics\val_mae_by_lag1Y.csv

============= SEGMENT-WISE MAE (TEST 2024) =============

Saved: kcet_ml_project\models\xgboost_stage3\diagnostics\test_mae_by_branch.csv
Saved: kcet_ml_project\models\xgboost_stage3\diagnostics\test_mae_by_examtype.csv
Saved: kcet_ml_project\models\xgboost_stage3\diagnostics\test_mae_by_round.csv
Saved: kcet_ml_project\models\xgboost_stage3\diagnostic

C:\Users\rahul\AppData\Local\Temp\ipykernel_27768\844501198.py:47: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
C:\Users\rahul\AppData\Local\Temp\ipykernel_27768\844501198.py:47: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
C:\Users\rahul\AppData\Local\Temp\ipykernel_27768\844501198.py:47:

In [22]:
# Cell: Display top 10 rows of all segmentation diagnostics CSVs

import pandas as pd
from pathlib import Path

DIAG_DIR = Path("kcet_ml_project/models/xgboost_stage3/diagnostics")

# List of all csvs to display
csv_files = [
    "val_mae_by_branch.csv",
    "val_mae_by_examtype.csv",
    "val_mae_by_round.csv",
    "val_mae_by_tier.csv",
    "val_mae_by_category_bin.csv",
    "val_mae_by_popularity.csv",
    "val_mae_by_histprimary.csv",
    "val_mae_by_lag1Y.csv",
    
    "test_mae_by_branch.csv",
    "test_mae_by_examtype.csv",
    "test_mae_by_round.csv",
    "test_mae_by_tier.csv",
    "test_mae_by_category_bin.csv",
    "test_mae_by_popularity.csv",
    "test_mae_by_histprimary.csv",
    "test_mae_by_lag1Y.csv",
]

print("=== SEGMENTATION DIAGNOSTICS — TOP 10 ROWS EACH ===\n")

for fname in csv_files:
    path = DIAG_DIR / fname
    print(f"\n\n----- {fname} -----")
    
    if not path.exists():
        print(f"❌ File not found: {path}")
        continue
    
    try:
        df = pd.read_csv(path)
        print(df.head(10).to_string(index=False))
    except Exception as e:
        print(f"⚠️ Could not load {fname}: {e}")

print("\n=== DONE ===")


=== SEGMENTATION DIAGNOSTICS — TOP 10 ROWS EACH ===



----- val_mae_by_branch.csv -----
 College_Branch_target_enc          MAE
             128107.587968 65625.062500
             142344.234961 60825.531250
             151063.562835 54554.992188
             159104.226142 53386.833333
             112051.020558 41790.768047
             152290.393984 40952.890625
             143691.134263 39720.614583
             152561.328320 38967.062500
             146230.987968 36896.546875
              85190.532760 36708.823438


----- val_mae_by_examtype.csv -----
 Exam_Type          MAE
         0 10843.766109
         1  4115.085164


----- val_mae_by_round.csv -----
 Round          MAE
     3 11434.014644
     0 11003.070195
     2 10536.365474
     1 10011.716375
     4  1282.129301


----- val_mae_by_tier.csv -----
 College_Tier_Numeric          MAE
                    5 12330.406483
                    3 12073.434073
                    4 11228.110554
                    2 10764.4434

In [24]:
# Fixed cell: Identify unstable branches + implement fallback blending & evaluate
import json, math, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

# Paths
PROJECT_ROOT = Path(".")
MODEL_DIR = PROJECT_ROOT / "kcet_ml_project" / "models" / "xgboost_stage3"
DIAG_DIR = MODEL_DIR / "diagnostics"
OUT_DIR = MODEL_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load wrapper & booster
wrapper_path = MODEL_DIR / "xgb_wrapper_stage3_final.joblib"
if not wrapper_path.exists():
    raise RuntimeError(f"Wrapper not found: {wrapper_path}")
wrapper = joblib.load(wrapper_path)
booster_path = Path(wrapper["booster_path"])
features = wrapper["features"]

bst = xgb.Booster()
bst.load_model(str(booster_path))
print("Loaded booster:", booster_path)

# Ensure X/y exist
try:
    Xtr = X_train_imputed.copy()
    Xv  = X_val_imputed.copy()
    Xt  = X_test_imputed.copy()
    ytr = np.asarray(y_train)
    yv  = np.asarray(y_val)
    yt  = np.asarray(y_test)
except NameError as e:
    raise RuntimeError("Required X/y (imputed) not found in memory. Run earlier cells to load them.") from e

# Load test global MAE from meta if available, else compute baseline
meta_path = MODEL_DIR / "xgb_final_meta_stage3.json"
global_test_mae = None
if meta_path.exists():
    try:
        with open(meta_path, "r") as f:
            meta = json.load(f)
        global_test_mae = meta.get("test_mae", None)
    except Exception as e:
        print("Warning: could not read meta file:", e)

# compute baseline predictions on test if needed
dtest = xgb.DMatrix(Xt[features])
pred_test_raw = bst.predict(dtest)
if global_test_mae is None:
    global_test_mae = mean_absolute_error(yt, pred_test_raw)
print(f"Global Test MAE (baseline): {global_test_mae:.2f}")

# Load per-branch MAE diagnostics
branch_diag_path = DIAG_DIR / "test_mae_by_branch.csv"
if not branch_diag_path.exists():
    raise RuntimeError(f"Diagnostics file not found: {branch_diag_path}")
branch_df = pd.read_csv(branch_diag_path)

# Heuristic threshold for instability
threshold = max(2.0 * global_test_mae, 50000.0)
print(f"Instability threshold set to: {threshold:.2f} (max(2*global_mae, 50k))")

# Select unstable branches
# ensure numeric
branch_df["College_Branch_target_enc"] = branch_df["College_Branch_target_enc"].astype(float)
unstable_branches = branch_df[branch_df["MAE"] > threshold]["College_Branch_target_enc"].astype(float).tolist()
unstable_branches = [float(x) for x in unstable_branches]
print(f"Found {len(unstable_branches)} unstable branch encodings (sample up to 10):", unstable_branches[:10])

# Build branch historical mean mapping from train+val (2020-2023)
X_full = pd.concat([Xtr.reset_index(drop=True), Xv.reset_index(drop=True)], axis=0).reset_index(drop=True)
y_full = np.concatenate([ytr, yv], axis=0)
df_full = X_full.copy()
df_full["y"] = y_full

branch_hist_group = df_full.groupby("College_Branch_target_enc")["y"]
branch_hist_mean = branch_hist_group.mean().to_dict()
# also store counts
branch_hist_count = branch_hist_group.count().to_dict()

# Save branch historical means
branch_hist_df = pd.DataFrame([
    {"College_Branch_target_enc": float(k), "hist_mean": float(v), "hist_count": int(branch_hist_count.get(k,0))}
    for k,v in branch_hist_mean.items()
])
branch_hist_df.to_csv(OUT_DIR / "branch_historical_mean.csv", index=False)
print("Saved branch_historical_mean.csv")

# Inference function with fallback blend
def predict_with_fallback(df, blend=0.7, min_hist_count=5):
    dmat = xgb.DMatrix(df[features])
    model_pred = bst.predict(dmat)
    out = model_pred.copy()
    branches = df["College_Branch_target_enc"].astype(float).values
    hist_counts = df.get("Historical_Count_Raw", pd.Series([0]*len(df))).values
    for i, (b, hc) in enumerate(zip(branches, hist_counts)):
        if (b in unstable_branches) or (pd.notna(hc) and hc < min_hist_count):
            hm = branch_hist_mean.get(float(b), None)
            if hm is None:
                if "Historical_Mean_Primary" in df.columns:
                    hm = float(df.iloc[i]["Historical_Mean_Primary"])
                else:
                    hm = float(np.mean(y_full))
            out[i] = blend * model_pred[i] + (1.0 - blend) * hm
    return out

# Evaluate on test: before/after
mae_before = mean_absolute_error(yt, pred_test_raw)
pred_test_after = predict_with_fallback(Xt, blend=0.7, min_hist_count=5)
mae_after = mean_absolute_error(yt, pred_test_after)
print(f"Test MAE BEFORE fallback: {mae_before:.2f}")
print(f"Test MAE AFTER fallback  (blend=0.7): {mae_after:.2f}")
print(f"Absolute improvement: {mae_before - mae_after:.2f}  ({(mae_before-mae_after)/mae_before*100:.2f}% reduction)")

# Save unstable branches and updated wrapper
unstable_path = OUT_DIR / "unstable_branches_stage3.json"
with open(unstable_path, "w") as f:
    json.dump({"unstable_branches": unstable_branches, "threshold": threshold}, f, indent=2)
print("Saved unstable branches to:", unstable_path)

# augment wrapper and save
new_wrapper = wrapper.copy()
new_wrapper.update({
    "fallback_blend": 0.7,
    "fallback_min_hist_count": 5,
    "unstable_branches_file": str(unstable_path.name),
    "branch_hist_mean_csv": "branch_historical_mean.csv"
})
joblib.dump(new_wrapper, OUT_DIR / "xgb_wrapper_stage3_final_with_fallback.joblib")
print("Saved updated wrapper with fallback info:", "xgb_wrapper_stage3_final_with_fallback.joblib")

# Show sample of unstable branches with hist counts/means and their test MAE
if unstable_branches:
    sample = []
    for b in unstable_branches[:20]:
        sample.append({
            "branch": float(b),
            "hist_count": int(branch_hist_count.get(b, 0)),
            "hist_mean": float(branch_hist_mean.get(b, float("nan"))),
            "test_mae": float(branch_df[branch_df["College_Branch_target_enc"]==b]["MAE"].iloc[0]) if (branch_df["College_Branch_target_enc"]==b).any() else None
        })
    print("\nSample unstable branches (up to 20):")
    print(pd.DataFrame(sample).to_string(index=False))
else:
    print("No unstable branches detected with the chosen threshold.")

# Recommendations
print("\nRECOMMENDATIONS:")
print("1) For unstable branches we applied a 70/30 model/historical blend in inference.")
print("2) Consider raising min_hist_count from 5 if still extreme errors.")
print("3) Flag predictions for unstable branches as 'low confidence' in UI and show historical_mean.")
print("4) Optionally produce a diagnostics CSV comparing pre/post MAE for only the unstable branches (I can run that next).")

print("\nCell complete.")


Loaded booster: kcet_ml_project\models\xgboost_stage3\xgb_booster_stage3_final_bestiter713.json
Global Test MAE (baseline): 26511.26
Instability threshold set to: 53022.52 (max(2*global_mae, 50k))
Found 43 unstable branch encodings (sample up to 10): [140699.3939842474, 101211.16732821411, 150360.7114210702, 20348.506337253177, 119589.96992123696, 25872.314637627547, 35853.423523059406, 136168.71640621728, 119700.73496061849, 106364.31328082464]
Saved branch_historical_mean.csv
Test MAE BEFORE fallback: 26511.26
Test MAE AFTER fallback  (blend=0.7): 27552.09
Absolute improvement: -1040.83  (-3.93% reduction)
Saved unstable branches to: kcet_ml_project\models\xgboost_stage3\unstable_branches_stage3.json
Saved updated wrapper with fallback info: xgb_wrapper_stage3_final_with_fallback.joblib

Sample unstable branches (up to 20):
       branch  hist_count     hist_mean      test_mae
140699.393984           2 106792.000000 118159.000000
101211.167328          43 150078.023256 109124.906250


In [25]:
# Adaptive per-branch fallback calibration on VAL (2023), apply to TEST (2024)
import json, math, numpy as np, pandas as pd, joblib, time
from pathlib import Path
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

ROOT = Path(".")
MODEL_DIR = ROOT / "kcet_ml_project" / "models" / "xgboost_stage3"
DIAG_DIR = MODEL_DIR / "diagnostics"

# load booster & wrapper
wrapper = joblib.load(MODEL_DIR / "xgb_wrapper_stage3_final.joblib")
bst = xgb.Booster(); bst.load_model(str(wrapper["booster_path"]))
features = wrapper["features"]

# load data (imputed) and y
Xtr = X_train_imputed.copy()   # train 2020-2022
Xv  = X_val_imputed.copy()     # val 2023
Xt  = X_test_imputed.copy()    # test 2024
ytr = np.asarray(y_train)
yv  = np.asarray(y_val)
yt  = np.asarray(y_test)

# load unstable branches list produced earlier (if exists)
unstable_path = MODEL_DIR / "unstable_branches_stage3.json"
if unstable_path.exists():
    unstable_list = json.load(open(unstable_path))["unstable_branches"]
else:
    unstable_list = []

print("Unstable branch count:", len(unstable_list))

# 1) compute historical means from TRAIN only (no leakage)
df_tr = Xtr.reset_index(drop=True).copy()
df_tr["y"] = ytr
train_branch_mean = df_tr.groupby("College_Branch_target_enc")["y"].mean().to_dict()
train_branch_count = df_tr.groupby("College_Branch_target_enc")["y"].count().to_dict()

# 2) compute model preds on VAL and TEST
dval = xgboost = None
dval = xgb.DMatrix(Xv[features])
dtst = xgb.DMatrix(Xt[features])
pred_val_model = bst.predict(dval)
pred_test_model = bst.predict(dtst)

# helper to get historical mean for a row (train-only mean), fallback to Historical_Mean_Primary, then global train mean
global_train_mean = np.mean(ytr)
def hist_for_row_from_train(idx, df_row):
    br = float(df_row["College_Branch_target_enc"])
    hm = train_branch_mean.get(br, None)
    if hm is not None:
        return float(hm)
    # try Historical_Mean_Primary from row
    if "Historical_Mean_Primary" in df_row.index:
        return float(df_row["Historical_Mean_Primary"])
    return float(global_train_mean)

# 3) For each unstable branch, calibrate best blend on VAL
blends = np.linspace(0.0, 1.0, 11)  # model weight from 0..1
per_branch_choice = {}  # branch -> {best_blend, val_mae_model, val_mae_best}

val_df = Xv.reset_index(drop=True).copy()
val_df["y_true"] = yv
val_df["pred_model"] = pred_val_model
# compute per-row hist_mean (from train)
val_df["hist_mean_train"] = val_df.apply(lambda r: hist_for_row_from_train(r.name, r), axis=1)

# operate only on unstable branches present in val
unstable_in_val = [b for b in unstable_list if float(b) in set(val_df["College_Branch_target_enc"].unique())]

for b in unstable_in_val:
    mask = val_df["College_Branch_target_enc"].astype(float) == float(b)
    sub = val_df[mask]
    if len(sub) == 0:
        continue
    y_true = sub["y_true"].values
    model_preds = sub["pred_model"].values
    hist_vals = sub["hist_mean_train"].values
    model_mae = mean_absolute_error(y_true, model_preds)
    best = {"blend": 1.0, "mae": model_mae}  # default keep model
    for w in blends:
        blended = w * model_preds + (1.0 - w) * hist_vals
        mae = mean_absolute_error(y_true, blended)
        if mae < best["mae"]:
            best = {"blend": float(w), "mae": float(mae)}
    per_branch_choice[float(b)] = {
        "val_model_mae": float(model_mae),
        "val_best_blend": best["blend"],
        "val_best_mae": best["mae"],
        "train_hist_count": int(train_branch_count.get(float(b), 0))
    }

# 4) Build final per-branch blend mapping:
# Accept the chosen blend only if it improves val MAE vs model by at least a small delta (e.g., 0.1%)
accepted_branch_blends = {}
for b, info in per_branch_choice.items():
    if info["val_best_mae"] < info["val_model_mae"] * 0.999:  # require at least 0.1% improvement
        accepted_branch_blends[b] = info["val_best_blend"]
    else:
        accepted_branch_blends[b] = 1.0  # keep model-only

print("Calibrated blends for unstable branches (sample up to 20):")
sample_items = list(accepted_branch_blends.items())[:20]
for b, w in sample_items:
    info = per_branch_choice[b]
    print(f"Branch {b}: model_mae={info['val_model_mae']:.1f}, best_blend_val={info['val_best_blend']:.2f}, chosen={w}")

# 5) Apply per-branch blend mapping to TEST
test_df = Xt.reset_index(drop=True).copy()
test_df["y_true"] = yt
test_df["pred_model"] = pred_test_model
# compute train-based hist_mean for test rows
test_df["hist_mean_train"] = test_df.apply(lambda r: hist_for_row_from_train(r.name, r), axis=1)

def predict_with_per_branch_blend(df):
    preds = df["pred_model"].values.copy()
    branches = df["College_Branch_target_enc"].astype(float).values
    hist_vals = df["hist_mean_train"].values
    for i, b in enumerate(branches):
        blend = accepted_branch_blends.get(float(b), 1.0)
        if blend < 1.0:
            preds[i] = blend * preds[i] + (1.0 - blend) * hist_vals[i]
    return preds

pred_test_adaptive = predict_with_per_branch_blend(test_df)
mae_before = mean_absolute_error(test_df["y_true"].values, test_df["pred_model"].values)
mae_after = mean_absolute_error(test_df["y_true"].values, pred_test_adaptive)
print(f"\nGlobal TEST MAE before adaptive blend: {mae_before:.2f}")
print(f"Global TEST MAE after  adaptive blend: {mae_after:.2f}")
print(f"Absolute delta: {mae_before - mae_after:.2f} ({(mae_before - mae_after)/mae_before*100:.2f}%)")

# 6) Evaluate only unstable-branch rows on Test
mask_unstable_test = test_df["College_Branch_target_enc"].astype(float).isin(list(accepted_branch_blends.keys()))
if mask_unstable_test.sum() > 0:
    mae_unstable_before = mean_absolute_error(test_df.loc[mask_unstable_test, "y_true"], test_df.loc[mask_unstable_test, "pred_model"])
    mae_unstable_after  = mean_absolute_error(test_df.loc[mask_unstable_test, "y_true"], pred_test_adaptive[mask_unstable_test.values])
    print(f"\nUnstable branches (n={mask_unstable_test.sum()}) TEST MAE before: {mae_unstable_before:.2f}, after: {mae_unstable_after:.2f}, delta: {mae_unstable_before - mae_unstable_after:.2f}")
else:
    print("\nNo unstable branches present in Test set to evaluate.")

# 7) Save mapping and updated wrapper
with open(MODEL_DIR / "per_branch_blend_stage3.json", "w") as f:
    json.dump(accepted_branch_blends, f, indent=2)
new_wrapper = wrapper.copy()
new_wrapper["per_branch_blend_file"] = "per_branch_blend_stage3.json"
joblib.dump(new_wrapper, MODEL_DIR / "xgb_wrapper_stage3_final_with_adaptive_fallback.joblib")
print("\nSaved per-branch blend mapping and updated wrapper.")

# 8) Print summary counts
num_blended = sum(1 for v in accepted_branch_blends.values() if v < 1.0)
print(f"Branches with an accepted blend <1.0: {num_blended} / {len(accepted_branch_blends)}")

print("\nDone.")


Unstable branch count: 43
Calibrated blends for unstable branches (sample up to 20):
Branch 140699.3939842474: model_mae=27432.3, best_blend_val=1.00, chosen=1.0
Branch 101211.16732821411: model_mae=13447.8, best_blend_val=1.00, chosen=1.0
Branch 150360.7114210702: model_mae=20154.8, best_blend_val=1.00, chosen=1.0
Branch 35853.423523059406: model_mae=4475.1, best_blend_val=1.00, chosen=1.0
Branch 136168.71640621728: model_mae=20615.6, best_blend_val=1.00, chosen=1.0
Branch 106364.31328082464: model_mae=776.9, best_blend_val=1.00, chosen=1.0
Branch 145997.1616535395: model_mae=19287.2, best_blend_val=0.50, chosen=0.5
Branch 151063.56283463913: model_mae=54555.0, best_blend_val=1.00, chosen=1.0
Branch 142344.2349606185: model_mae=60825.5, best_blend_val=1.00, chosen=1.0
Branch 119794.23496061849: model_mae=9409.8, best_blend_val=1.00, chosen=1.0
Branch 122935.41997749628: model_mae=9276.7, best_blend_val=0.70, chosen=0.7000000000000001
Branch 112051.02055833898: model_mae=41790.8, best_

In [26]:
# Packaging cell: create versioned package with artifacts, checksums, and inference template
import os, json, shutil, hashlib, time
from pathlib import Path
import joblib
import numpy as np

ROOT = Path(".")
MODEL_DIR = ROOT / "kcet_ml_project" / "models" / "xgboost_stage3"

timestamp = time.strftime("%Y%m%dT%H%M%S")
pkg_name = f"package_v1_{timestamp}"
PKG_DIR = MODEL_DIR / pkg_name
PKG_DIR.mkdir(parents=True, exist_ok=True)

# Files to consider copying (common names in this notebook)
candidates = [
    "xgb_booster_stage3_final_bestiter713.json",
    "xgb_booster_stage3_final_bestiter0.json",
    "xgb_booster_stage3_optuna_best.json",
    "xgb_wrapper_stage3_final_with_adaptive_fallback.joblib",
    "xgb_wrapper_stage3_final_with_fallback.joblib",
    "xgb_wrapper_stage3_final.joblib",
    "xgb_wrapper_stage3_optuna_best.joblib",
    "xgb_optuna_best_params_stage3.json",
    "xgb_train_meta_stage3.json",
    "xgb_final_meta_stage3.json",
    "xgb_train_meta_stage3.json",
    "per_branch_blend_stage3.json",
    "unstable_branches_stage3.json",
    "branch_historical_mean.csv",
    "shap_val.npy",
    "shap_test.npy",
    "shap_mean_abs_val.csv",
    "shap_mean_abs_test.csv",
    "shap_top15_val.png",
    "shap_top15_test.png",
    "diagnostics/test_mae_by_branch.csv",
    "diagnostics/test_mae_by_examtype.csv",
    "diagnostics/test_mae_by_round.csv",
    "diagnostics/test_mae_by_tier.csv",
    "diagnostics/test_mae_by_category_bin.csv",
    "diagnostics/test_mae_by_popularity.csv",
    "diagnostics/test_mae_by_histprimary.csv",
    "diagnostics/test_mae_by_lag1Y.csv",
    "diagnostics/val_mae_by_branch.csv",
    "diagnostics/val_mae_by_examtype.csv",
    "diagnostics/val_mae_by_round.csv",
    "diagnostics/val_mae_by_tier.csv",
    "diagnostics/val_mae_by_category_bin.csv",
    "diagnostics/val_mae_by_popularity.csv",
    "diagnostics/val_mae_by_histprimary.csv",
    "diagnostics/val_mae_by_lag1Y.csv",
    "imputation_map_stage3.json",
    "stage2_feature_config.json",
    "stage2_summary.json",
    "xgb_optuna_best_params_stage3.json"
]

# Resolve existing path names and copy them
copied = []
for name in candidates:
    src = MODEL_DIR / name
    # also allow top-level files or nested 'diagnostics' already included
    if not src.exists():
        # try without diagnostics prefix
        alt = MODEL_DIR / name.split("diagnostics/")[-1]
        if alt.exists():
            src = alt
    if src.exists():
        dest = PKG_DIR / src.name
        shutil.copy2(src, dest)
        copied.append(dest)
    else:
        # search recursively in MODEL_DIR for matching basename
        matches = list(MODEL_DIR.rglob(Path(name).name))
        if matches:
            src = matches[0]
            dest = PKG_DIR / src.name
            shutil.copy2(src, dest)
            copied.append(dest)
        # else ignore missing (not all notebooks produce all files)

# Save features.json (locked_features variable) if available
features_file = PKG_DIR / "features.json"
try:
    import json as _json
    feat = locked_features  # from notebook state
    with open(features_file, "w") as f:
        _json.dump({"features": feat}, f, indent=2)
    copied.append(features_file)
except Exception as e:
    print("Warning: locked_features not found in memory; skipping features.json")

# Save imputer map if in memory
try:
    if 'imputation_map' in globals():
        with open(PKG_DIR / "imputation_map_stage3.json", "w") as f:
            json.dump(imputation_map, f, indent=2)
        copied.append(PKG_DIR / "imputation_map_stage3.json")
except Exception:
    pass

# Save a small stage3 meta summarizing final metrics and artifacts
meta_out = {
    "packaged_at": timestamp,
    "package_name": pkg_name,
    "artifacts": [p.name for p in copied]
}
# try to include final meta if exists
final_meta_path = MODEL_DIR / "xgb_final_meta_stage3.json"
if final_meta_path.exists():
    try:
        with open(final_meta_path) as f:
            meta_final = json.load(f)
        meta_out["final_meta"] = meta_final
    except Exception:
        pass

with open(PKG_DIR / "stage3_meta_package.json", "w") as f:
    json.dump(meta_out, f, indent=2)
copied.append(PKG_DIR / "stage3_meta_package.json")

# Compute SHA256 checksums for all packaged files
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

checks = {}
for p in copied:
    try:
        checks[p.name] = sha256_file(p)
    except Exception:
        checks[p.name] = None

with open(PKG_DIR / "checksums.sha256", "w") as f:
    for name, h in checks.items():
        if h:
            f.write(f"{h}  {name}\n")
        else:
            f.write(f"NONE  {name}\n")
copied.append(PKG_DIR / "checksums.sha256")

# Write inference_template.py
inference_code = f'''"""
Inference template for packaged XGBoost model (Stage 3).
Place this alongside the package files and call predict_df(df) with a pandas DataFrame.
It expects:
 - xgb_wrapper_stage3_final_with_adaptive_fallback.joblib (wrapper)
 - xgb_booster_stage3_final_bestiter713.json (booster JSON)
 - per_branch_blend_stage3.json
 - branch_historical_mean.csv
"""

import json
from pathlib import Path
import joblib
import xgboost as xgb
import pandas as pd
import numpy as np

PKG_DIR = Path(__file__).resolve().parent

# Load wrapper and booster
wrapper = joblib.load(PKG_DIR / "xgb_wrapper_stage3_final_with_adaptive_fallback.joblib")
booster_path = wrapper.get("booster_path")
features = wrapper.get("features")
bst = xgb.Booster()
bst.load_model(str(PKG_DIR / Path(booster_path).name))

# Load per-branch blend mapping and branch hist means
blend_map = {{}}
per_branch_file = PKG_DIR / "per_branch_blend_stage3.json"
if per_branch_file.exists():
    with open(per_branch_file) as f:
        blend_map = json.load(f)
branch_hist = {{}}
hist_csv = PKG_DIR / "branch_historical_mean.csv"
if hist_csv.exists():
    bh = pd.read_csv(hist_csv)
    branch_hist = dict(zip(bh["College_Branch_target_enc"].astype(float), bh["hist_mean"]))

def predict_df(df, blend_default=1.0, min_hist_count=5):
    """
    df: pandas DataFrame including the same features used for training.
    Returns: numpy array of final predictions (after adaptive fallback)
    """
    # Ensure features present
    X = df.copy()
    missing = [c for c in features if c not in X.columns]
    if missing:
        raise ValueError(f"Missing features in input DataFrame: {{missing}}")
    dmat = xgb.DMatrix(X[features])
    preds = bst.predict(dmat)
    # compute hist mean per-row (from packaged branch_hist fallback)
    branch_vals = X["College_Branch_target_enc"].astype(float).values
    hist_vals = np.array([branch_hist.get(float(b), np.nan) for b in branch_vals])
    # apply per-branch blends where available; else leave model pred
    final = preds.copy()
    for i, b in enumerate(branch_vals):
        b_key = str(float(b))
        blend = 1.0
        if b_key in blend_map:
            blend = float(blend_map[b_key])
        elif float(b) in blend_map:
            blend = float(blend_map[float(b)])
        if blend < 1.0:
            hm = hist_vals[i]
            if pd.isna(hm):
                # fallback to Historical_Mean_Primary if present
                if "Historical_Mean_Primary" in X.columns:
                    hm = float(X.iloc[i]["Historical_Mean_Primary"])
                else:
                    hm = float(np.nanmean(list(branch_hist.values())))  # last resort
            final[i] = blend * preds[i] + (1.0 - blend) * hm
    return np.asarray(final)

# Example usage:
# import pandas as pd
# df = pd.read_csv("some_input.csv")
# preds = predict_df(df)
# print(preds[:10])
'''

with open(PKG_DIR / "inference_template.py", "w") as f:
    f.write(inference_code)
copied.append(PKG_DIR / "inference_template.py")

# README
readme = f"""KCET Stage3 XGBoost packaging
Package: {pkg_name}
Created: {timestamp}

Contents:
- model booster(s), wrappers, per-branch blending map, branch historical means
- SHAP artifacts, diagnostics, and checksums

Usage:
- Copy package folder to your deployment host.
- Use inference_template.py as a starting point for loading model and running predictions.
- Validate predictions on your infra before serving.

Notes:
- This package includes an adaptive per-branch blend mapping calibrated on Val(2023). Keep that file with the package.
"""

with open(PKG_DIR / "README.md", "w") as f:
    f.write(readme)
copied.append(PKG_DIR / "README.md")

print("Packaged files into:", PKG_DIR)
print("Files included:")
for p in copied:
    print(" -", p.name)
print("\nChecksums file saved as checksums.sha256 in package dir.")
print("\nPackaging complete.")


Packaged files into: kcet_ml_project\models\xgboost_stage3\package_v1_20251119T100521
Files included:
 - xgb_booster_stage3_final_bestiter713.json
 - xgb_booster_stage3_final_bestiter0.json
 - xgb_booster_stage3_optuna_best.json
 - xgb_wrapper_stage3_final_with_adaptive_fallback.joblib
 - xgb_wrapper_stage3_final_with_fallback.joblib
 - xgb_wrapper_stage3_final.joblib
 - xgb_wrapper_stage3_optuna_best.joblib
 - xgb_optuna_best_params_stage3.json
 - xgb_train_meta_stage3.json
 - xgb_final_meta_stage3.json
 - xgb_train_meta_stage3.json
 - per_branch_blend_stage3.json
 - unstable_branches_stage3.json
 - branch_historical_mean.csv
 - shap_val.npy
 - shap_test.npy
 - shap_mean_abs_val.csv
 - shap_mean_abs_test.csv
 - shap_top15_val.png
 - shap_top15_test.png
 - test_mae_by_branch.csv
 - test_mae_by_examtype.csv
 - test_mae_by_round.csv
 - test_mae_by_tier.csv
 - test_mae_by_category_bin.csv
 - test_mae_by_popularity.csv
 - test_mae_by_histprimary.csv
 - test_mae_by_lag1Y.csv
 - val_mae_by_b

In [28]:
# Fixed cell: compute percentage-style accuracies (model-only vs adaptive blend)
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import r2_score, mean_absolute_error

MODEL_DIR = Path("kcet_ml_project/models/xgboost_stage3")

# --- Helpers ---
def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    # avoid divide by zero
    denom = np.maximum(np.abs(y_true), 1e-6)
    return np.mean(np.abs((y_true - y_pred) / denom))

def mae_relative_accuracy(mae_val, y_true):
    return 1.0 - (mae_val / np.mean(np.abs(y_true)))

def print_accuracy(title, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape_val = mape(y_true, y_pred)
    print(f"\n=== {title} ===")
    print(f"MAE: {mae:,.2f}")
    print(f"R²: {r2:.6f} -> Accuracy (R²-based): {r2*100:.2f}%")
    print(f"MAPE: {mape_val*100:.4f}% -> Accuracy (MAPE-based): {(1-mape_val)*100:.2f}%")
    print(f"MAE Relative Accuracy: {mae_relative_accuracy(mae, y_true)*100:.2f}%")

# --- Load required objects from notebook state ---
# Expect these to exist in memory: bst (xgboost Booster), locked_features (list), X_train_imputed, X_val_imputed, X_test_imputed, y_train, y_val, y_test
try:
    bst  # booster
    locked_features
    Xtr = X_train_imputed
    Xv  = X_val_imputed
    Xt  = X_test_imputed
    ytr = np.asarray(y_train)
    yv  = np.asarray(y_val)
    yt  = np.asarray(y_test)
except NameError as e:
    raise RuntimeError("Required objects missing from memory. Ensure bst, locked_features, X_*_imputed and y_* exist.") from e

# --- Model-only predictions ---
import xgboost as xgb
dval = xgb.DMatrix(Xv[locked_features])
dtest = xgb.DMatrix(Xt[locked_features])
val_pred_model = bst.predict(dval)
test_pred_model = bst.predict(dtest)

# --- Adaptive blend mapping and branch hist means (train-based) ---
per_branch_file = MODEL_DIR / "per_branch_blend_stage3.json"
branch_hist_csv = MODEL_DIR / "branch_historical_mean.csv"

if per_branch_file.exists():
    per_branch_blend = json.load(open(per_branch_file))
    # keys may be strings or numbers; normalize to float->float mapping
    blend_map = {}
    for k,v in per_branch_blend.items():
        try:
            blend_map[float(k)] = float(v)
        except Exception:
            # try if already float-like
            blend_map[k] = float(v)
else:
    blend_map = {}

if branch_hist_csv.exists():
    bh = pd.read_csv(branch_hist_csv)
    branch_hist_map = {float(r["College_Branch_target_enc"]): float(r["hist_mean"]) for _, r in bh.iterrows()}
else:
    branch_hist_map = {}

# fallback train mean & function to get hist mean per row
global_train_mean = float(np.mean(ytr))
def hist_mean_for_row(row):
    b = float(row["College_Branch_target_enc"])
    hm = branch_hist_map.get(b, None)
    if hm is not None and not np.isnan(hm):
        return float(hm)
    # fallback to Historical_Mean_Primary column if present
    if "Historical_Mean_Primary" in row.index:
        return float(row["Historical_Mean_Primary"])
    return global_train_mean

# --- Build adaptive predictions (apply per-branch blends where blend<1.0) ---
def build_adaptive_preds(df, model_preds):
    # df must have column 'College_Branch_target_enc'
    preds = np.array(model_preds, copy=True)
    for i, row_idx in enumerate(df.index):
        row = df.loc[row_idx]
        b = float(row["College_Branch_target_enc"])
        blend = blend_map.get(b, 1.0)
        if blend < 1.0:
            hm = hist_mean_for_row(row)
            preds[i] = blend * preds[i] + (1.0 - blend) * float(hm)
    return preds

val_pred_adaptive = build_adaptive_preds(Xv, val_pred_model)
test_pred_adaptive = build_adaptive_preds(Xt, test_pred_model)

# --- Print accuracies ---
print_accuracy("Validation 2023 (Model Only)", yv, val_pred_model)
print_accuracy("Validation 2023 (Adaptive Blend)", yv, val_pred_adaptive)

print_accuracy("Test 2024 (Model Only)", yt, test_pred_model)
print_accuracy("Test 2024 (Adaptive Blend)", yt, test_pred_adaptive)

# Also print a tiny summary table showing count of blended rows and number of branches used
num_blended_test = sum(1 for b in Xt["College_Branch_target_enc"].astype(float).values if float(b) in blend_map and blend_map[float(b)] < 1.0)
print(f"\nTest rows blended (count): {num_blended_test} / {len(Xt)}")
num_branches_blended = sum(1 for v in set(blend_map.values()) if v < 1.0)
print(f"Branches with blend <1.0: {num_branches_blended} (in blend map)")

# End of cell



=== Validation 2023 (Model Only) ===
MAE: 10,629.98
R²: 0.901274 -> Accuracy (R²-based): 90.13%
MAPE: 16.7966% -> Accuracy (MAPE-based): 83.20%
MAE Relative Accuracy: 86.97%

=== Validation 2023 (Adaptive Blend) ===
MAE: 10,628.76
R²: 0.901284 -> Accuracy (R²-based): 90.13%
MAPE: 16.7960% -> Accuracy (MAPE-based): 83.20%
MAE Relative Accuracy: 86.98%

=== Test 2024 (Model Only) ===
MAE: 26,511.26
R²: 0.695145 -> Accuracy (R²-based): 69.51%
MAPE: 26.3248% -> Accuracy (MAPE-based): 73.68%
MAE Relative Accuracy: 75.81%

=== Test 2024 (Adaptive Blend) ===
MAE: 26,503.41
R²: 0.695368 -> Accuracy (R²-based): 69.54%
MAPE: 26.3217% -> Accuracy (MAPE-based): 73.68%
MAE Relative Accuracy: 75.82%

Test rows blended (count): 124 / 71626
Branches with blend <1.0: 7 (in blend map)
